# Build Clean — pipeline de limpieza y análisis de datasets de anime

Pipeline completo desde los volcados originales de AniList y MyAnimeList hasta las tablas
analíticas y los gráficos. Cada celda de código hace **una sola cosa** y va precedida de un
markdown que explica el criterio aplicado y cómo funciona el código.

## Criterio general del pipeline

- **No se inventan valores**: los nulos permanecen nulos.
- **No se descarta nada sin dejar evidencia**: toda eliminación o ajuste queda registrado
  en `_report.json`.
- **Los valores imposibles se marcan, no se corrigen**: se añade una bandera al lado del
  dato original y quien analiza decide.
- **Una comprobación que no se puede hacer no es una comprobación superada**: si falta el
  dato de referencia, la bandera queda nula, no `False`.
- **Las reglas deben ser verificables** a partir del propio dataset o de la documentación
  de la fuente. Lo que no se puede verificar se documenta como supuesto.
- **Todo muestreo se declara** en la salida y en el título del gráfico correspondiente.

## Estructura

| Bloque | Contenido | Ejecuta |
|---|---|---|
| **1. Configuración** | Rutas, constantes, helpers de escritura y normalización | Sí (instantáneo) |
| **2. Etapa 1 — Catálogo** | Lectura de AniList, duplicados, fechas, despliegue de JSON | Solo define |
| **3. Etapa 2 — Perfiles** | Snapshot, fechas, validaciones, favoritos, amistades, texto | Solo define |
| **4. Etapa 3 — Interacciones** | Streaming de 968M de filas y agregados por anime | Solo define |
| **5. Etapa 4 — Informe** | Volcado de `_report.json` y cobertura de metadatos | Solo define |
| **6. Auditoría** | Reparto de nulos por tabla y política de reparación | Sí |
| **7. Configuración analítica** | Umbrales, helpers de gráfico y lectores | Sí |
| **8. Gráficos exploratorios** | Seis vistas sobre catálogo y usuarios | Sí |
| **9. Preparación analítica** | Tablas derivadas en `clean/analysis/` | Sí (~8 min) |
| **10. Análisis** | Perfiles, géneros, popularidad y segmentación | Sí |

Los bloques 8 y 10 usan además los volcados de otros años que hay en
`D:\Codigos\Visualizacion`: el censo de usuarios de 2023 para los cruces demográficos y
los nueve cortes de `panel_temporal/` para medir la misma curva en momentos distintos.

Los bloques 2 a 5 **solo definen funciones**: la llamada a `main()` está comentada a
propósito, porque la etapa 3 recorre 968 millones de filas y tarda horas. Ejecutar el
notebook entero de arriba abajo no lanza el pipeline pesado.

Los bloques 6 a 10 trabajan sobre los parquet ya generados en `clean/` y se pueden
ejecutar directamente.

## Ficheros de entrada

| Fichero | Tamaño | Contenido |
|---|---|---|
| `anilist_anime_data_complete.csv` | 438 MB | Catálogo de AniList, 20.416 títulos |
| `profile.parquet` | 1,9 GB | 18,2 millones de perfiles de MyAnimeList |
| `ulist_anime.parquet` | 12,5 GB | 968 millones de filas usuario × anime |
| `ulist_anime_fetch.parquet` | 99 MB | Registro de la descarga de listas |


# Correcciones aplicadas en esta revisión

Resumen de los defectos detectados y corregidos, para que el notebook se pueda ejecutar
de principio a fin y los resultados sean interpretables.

**Errores que impedían ejecutar el notebook**

1. La celda de preparación analítica cargaba `interacciones.parquet` entero con
   `pd.read_parquet` (527M de filas, 8.4 GB): agota la memoria. Ahora se agrega por
   row group con `iter_interacciones()`.
2. Las tablas analíticas se construían sin las columnas que usaban las celdas
   siguientes: `df_users` no incluía `anime_completed` y `df_anime_resumen` no incluía
   `title` ni `popularity`, así que las celdas de análisis fallaban con `KeyError`.
   `anime.parquet` no tiene columna `title`: se deriva de `title_english` →
   `title_romaji` → `title_native`.
3. `np.polyfit` recibía dos series con `dropna()` independiente, de longitudes
   distintas. Ahora se ajusta sobre las filas donde ambas variables existen.

**Errores de limpieza**

4. `comm` trae la cadena literal `"null"` en el 98% de los perfiles, y el filtro solo
   descartaba cadenas vacías: `user_texto.parquet` se quedaba con los 18.2M de usuarios
   (1.2 GB) en lugar de los que tienen texto real. Se añade una lista de centinelas.
5. Las banderas `*_invalid` se calculaban como `valor > 10`, y un nulo daba `False`,
   es decir "válido". Ahora son booleanas con nulos.
6. El tamaño de los acumuladores de la etapa 3 estaba fijado en 70000: las filas con
   `anime_id` mayor se descartaban en silencio. Ahora se deriva de las estadísticas
   del propio parquet.
7. La cohorte se filtraba con `Index.get_indexer`, que falla si hay usuarios repetidos.
   Se usa `isin` sobre valores únicos.
8. `score_zero` mezclaba puntuación ausente y puntuación cero. Se separan
   `score_missing` y `score_out_of_range`.

**Errores de análisis**

8b. `anime_mean` vale 0 cuando el perfil no ha puntuado ningún anime (13.9% de los
    usuarios activos). Tomarlo como nota real hundía todas las medias por segmento:
    8.23 frente a 7.09. Ahora se trata como ausente.

9. "Top 10 mejor valorados" no exigía un mínimo de audiencia, así que devolvía títulos
   con dos o tres valoraciones. Se aplica un umbral explícito y se indica en el gráfico.
10. "Joyas ocultas" ordenaba por `nota × log(audiencia)`, que premia a los títulos más
    vistos: devolvía los éxitos masivos. Ahora exige nota alta **y** audiencia por debajo
    de un percentil del catálogo.
11. "Evolución en el tiempo" dibujaba diez barras verticales sin ninguna serie temporal.
    Ahora traza la nota media por año de estreno con los títulos destacados encima.
12. Los gráficos por edad y por género leían solo 20 de los 924 bloques de interacciones
    sin decirlo. El muestreo aparece ahora en el título del gráfico.
13. Los "hallazgos" del análisis de perfiles estaban escritos a mano en los `print`
    («abandonan ~42 animes», «6.8-7.3»). Ahora se calculan de los datos.
14. La edad se calculaba con el año 2026 fijo en el código; se toma del snapshot.
15. `anime_stats_globales.parquet` se calcula sobre **todas** las filas del origen,
    mientras que `interacciones.parquet` contiene solo la cohorte. Queda documentado
    en el informe porque las dos tablas no son comparables fila a fila.

**Mejoras con los datasets de otros años (2018-2025)**

20. **El mapeo de códigos de género estaba invertido.** El notebook usaba `1: Femenino,
    2: Masculino` con la advertencia de que era una interpretación no documentada. El volcado
    de 2023 trae el género como texto y comparte usuarios con este: cruzando 222.619 nombres,
    el código 1 es *Male* en el 100% de los casos y el 2 es *Female* en el 99,9%. El gráfico
    8.4 llevaba las barras al revés.
21. **Joyas ocultas restringidas a primeras temporadas.** Una cuarta temporada con nota alta
    y poca audiencia no es un descubrimiento: es el público residual de una serie conocida, y
    su nota está inflada porque solo la puntúa quien llegó hasta ahí. El filtro sale de
    `anime_relaciones.parquet` (relaciones `PREQUEL` y `PARENT`), no de buscar «Season 2» en
    el título. Descarta 6.584 de 19.302 títulos y cambia la lista por completo.
22. **Los gráficos 8.3 y 8.4 pasan del 2% al 100% de los datos.** Leían 20 de los 924 bloques
    de interacciones; ahora recorren el censo del volcado de 2023, que cabe entero. Y cambian
    de forma: con los datos completos, *Action* domina en todos los grupos, así que el gráfico
    de «género dominante» solo medía el tamaño de cada grupo. Ahora muestran la desviación
    frente al reparto general, que es donde están las diferencias reales.
23. **El gráfico 8.5 comprueba su propio confusor.** Antes advertía de que la subida podía ser
    supervivencia; ahora dibuja la curva desde cuatro cortes distintos y se ve que las tres
    con nota publicada por MAL se solapan punto por punto.
24. **Fuera el doble eje de 8.5.** Superponía nota y número de estrenos en dos escalas sobre
    el mismo plano, lo que insinúa una correlación que el gráfico no mide. Ahora son dos
    paneles apilados.
25. **8.2 valida el ranking contra la otra plataforma.** Mostraba solo el top de AniList; ahora
    pone los dos rankings juntos y colorea los coincidentes. Salen 8 de 10.
26. **10.5 deja de mezclar orígenes.** Cruzaba popularidad de AniList (agosto de 2026) con nota
    de MyAnimeList (febrero de 2026). Ahora las dos variables salen de la misma pasada. La
    correlación pasa de 0,539 a 0,525: la conclusión aguanta.
27. **10.4 avisa de la diferencia de taxonomía.** Usa los géneros de AniList, que solo comparte
    14 etiquetas con los de MyAnimeList. *Thriller* y *Suspense* son casi el mismo concepto en
    los dos vocabularios.

**Otros**

16. Acentos corregidos en títulos y etiquetas de los gráficos (venían con `?` por una
    conversión de codificación).
17. Rutas, umbrales y helpers unificados en una sola celda de configuración en vez de
    repetirse en cada celda.
18. `matplotlib.use("Agg")` impedía ver los gráficos en el notebook; ahora se guardan
    y se muestran.
19. El bloque de "joyas ocultas" estaba duplicado en dos celdas escribiendo el mismo PNG.


# Estructura de las tablas base y cómo se relacionan

El pipeline se construye sobre tres tablas principales:

1. `anime.parquet`
   - Es la tabla base del catálogo de anime normalizado desde AniList.
   - La clave principal es `mal_id`.
   - A partir de aquí se vinculan géneros, tags, rankings, personajes, relaciones y recomendaciones.

2. `users.parquet`
   - Es la tabla base del perfil del usuario.
   - La clave principal es `username`.
   - Se usa para relacionar cada usuario con sus favoritos, amigos, texto libre y sus interacciones.

3. `interacciones.parquet`
   - Es la tabla de hechos del sistema.
   - Contiene la relación usuario-anime: `username` + `anime_id`, junto con estado, score, fechas, episodios vistos y otros atributos de la lista.

A partir de esas tablas base se crean tablas derivadas en formato largo o aplanado:

- `anime_generos.parquet` -> unión por `mal_id`
- `anime_tags.parquet` -> unión por `mal_id`
- `anime_personajes_anilist.parquet` -> unión por `mal_id`
- `anime_rankings.parquet` -> unión por `mal_id`
- `anime_relaciones.parquet` -> unión por `mal_id_from` y `mal_id_to`
- `anime_recomendaciones.parquet` -> unión por `mal_id_from` y `mal_id_to`
- `user_favoritos_anime.parquet` -> unión por `username`
- `user_amigos.parquet` -> unión por `username`
- `user_texto.parquet` -> unión por `username`

Las uniones clave del análisis son:

- `users.username` con `interacciones.username`
- `anime.mal_id` con `anime_stats_globales.anime_id`
- `anime.mal_id` con las tablas de géneros, tags, personajes y rankings
- `users.username` con los datos de favoritos y amistades

La idea general es mantener una base central de entidades y luego expandirla en tablas 1:N que describen atributos o relaciones complementarias, sin duplicar la información de la tabla principal.


# PARTE 1: CONFIGURACIÓN

# CONFIGURACION INICIAL

Esta celda prepara el entorno de trabajo. Se definen:
- rutas de entrada y salida,
- nombres de archivos fuente,
- reglas de negocio que se usarán durante la limpieza,
- helpers compartidos para logging, escritura parquet y normalización.

Criterio del pipeline:
1) no inventar valores,
2) registrar todo lo que se descarta o corrige,
3) preservar el dato original y marcar invalidaciones cuando exista una regla verificable,
4) operar sobre estructura de datos robusta para datos masivos.


## 1.1 Rutas, constantes y diccionario del informe

Esta celda no transforma nada: solo fija el contrato con el que trabaja el resto del
notebook. Se separa del código de limpieza para que cambiar una ruta o un umbral no
obligue a tocar ninguna función.

**Qué define**

| Bloque | Contenido | Por qué está aquí |
|---|---|---|
| Rutas | `BASE`, `DIR_USERS`, `DIR_ANI`, `OUT` | Un único punto donde cambiar la ubicación de los datos |
| Ficheros fuente | `F_PROFILE`, `F_ULIST`, `F_FETCH`, `F_ANILIST` | Los cuatro únicos ficheros de entrada del pipeline |
| Reglas de negocio | `STATUS_LABELS`, `MAX_STATUS_CODE`, `SCORE_MIN`, `SCORE_MAX` | Los códigos de estado y la escala de puntuación de MyAnimeList |
| Centinelas | `NULL_SENTINELS` | Cadenas que la fuente escribe en lugar de un nulo real |
| Informe | `REPORT` | Acumula el recuento de cada etapa; se vuelca a `_report.json` al final |

**Criterio sobre los centinelas.** La fuente no siempre escribe nulos: se ha comprobado
sobre `profile.parquet` que la columna `comm` contiene la cadena literal `"null"` en el
98% de las filas. Tratar esas cadenas como texto real hacía que `user_texto.parquet`
conservara los 18,2 millones de perfiles (1,2 GB) en lugar de los que tienen texto de
verdad. La lista `NULL_SENTINELS` centraliza esos valores para que la decisión sea única
y revisable, no una condición repetida en cada celda.

**Sobre `STATUS_LABELS`.** El código 5 no existe en MyAnimeList: la numeración salta de 4
(*dropped*) a 6 (*plan to watch*). El diccionario refleja esa discontinuidad y por eso
`MAX_STATUS_CODE` vale 6 aunque solo haya cinco estados válidos.


In [ ]:
import json, os, re, time, html
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

BASE = r"D:\Codigos\Visualizacion\dataset 2026"
DIR_USERS = os.path.join(BASE, "kaggle_download_users_huge")
DIR_ANI = os.path.join(BASE, "kaggle_download_anilist")
OUT = os.path.join(BASE, "clean")

F_PROFILE = os.path.join(DIR_USERS, "profile.parquet")
F_ULIST = os.path.join(DIR_USERS, "ulist_anime.parquet")
F_FETCH = os.path.join(DIR_USERS, "ulist_anime_fetch.parquet")
F_ANILIST = os.path.join(DIR_ANI, "anilist_anime_data_complete.csv")

COMPRESSION = "zstd"
STATUS_LABELS = {1: "watching", 2: "completed", 3: "on_hold", 4: "dropped", 6: "plan_to_watch"}
MAX_STATUS_CODE = 6
SCORE_MIN, SCORE_MAX = 1, 10

# Centinelas de texto que la fuente escribe como cadena literal en vez de como nulo.
# Verificado sobre profile.parquet: la columna `comm` vale la cadena "null" en el 98% de las filas.
NULL_SENTINELS = {"", "null", "NULL", "None", "nan", "NaN", "-", "n/a", "N/A"}

REPORT = {"generated_at_utc": datetime.now(timezone.utc).isoformat(), "stages": {}}

os.makedirs(OUT, exist_ok=True)
print(f"Directorio de salida: {OUT}")


## 1.2 Helpers de registro y escritura

Dos funciones que usan todas las etapas. Se aíslan aquí porque son infraestructura, no
lógica de limpieza.

**`log(msg)`** — imprime con marca de tiempo y `flush=True`. El `flush` importa: la etapa 3
tarda horas y sin él la salida se quedaría en el búfer y no se vería el progreso.

**`write(df_or_tbl, name, schema=None)`** — escribe un parquet en `OUT`, informa por
pantalla de filas, columnas y tamaño, y **registra el resultado en `REPORT["outputs"]`**.
Esa anotación automática es la que hace que el informe final no dependa de que nadie se
olvide de apuntar una salida.

El parámetro `schema` cubre un caso concreto: si un `DataFrame` llega vacío,
`pa.Table.from_pandas` produce un parquet de **cero columnas**, y cualquier lectura
posterior por nombre de columna falla. Pasando el esquema, la tabla vacía conserva su
estructura.

Criterio general: **nada se escribe sin quedar registrado**. Si un fichero aparece en
`clean/`, aparece también en `_report.json` con su número de filas.


In [ ]:
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def write(df_or_tbl, name, schema=None, **kw):
    """Escribe parquet y registra el resultado en REPORT.

    `schema` permite fijar el esquema cuando el DataFrame puede venir vacio: sin el,
    un DataFrame sin filas produce un parquet de 0 columnas que rompe las lecturas
    posteriores por nombre de columna.
    """
    path = os.path.join(OUT, name)
    if isinstance(df_or_tbl, pa.Table):
        tbl = df_or_tbl
    elif schema is not None:
        df = df_or_tbl
        if df.empty:
            df = pd.DataFrame({f.name: pd.Series(dtype=f.type.to_pandas_dtype()) for f in schema})
        tbl = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
    else:
        tbl = pa.Table.from_pandas(df_or_tbl, preserve_index=False)
    pq.write_table(tbl, path, compression=COMPRESSION, **kw)
    size = os.path.getsize(path) / 1024**2
    log(f"    -> {name:<38} {tbl.num_rows:>12,} filas  {tbl.num_columns:>3} cols  {size:8.1f} MB")
    REPORT.setdefault("outputs", {})[name] = {
        "rows": tbl.num_rows,
        "columns": tbl.num_columns,
        "size_mb": round(size, 2),
    }
    return path


## 1.3 Helpers de normalización

Cinco funciones puras (sin efectos secundarios) que resuelven los problemas recurrentes
de estas fuentes de datos.

**`explode_list_column(list_array, id_values, value_name, id_name)`**
Convierte una columna de listas de Arrow en formato largo *preservando la posición
original* en la columna `pos`. Lo hace con `numpy` en lugar de `pandas.explode` porque
opera sobre 18 millones de filas: se calculan las longitudes de cada lista, se repite el
identificador padre con `np.repeat` y se aplana el array de una sola vez. El orden
importa: en una lista de favoritos, la posición 0 es el favorito principal.

**`parse_json_safe(s)`**
Varias columnas de AniList vienen como JSON dentro de una celda de texto. Un JSON roto no
puede interrumpir el proceso ni confundirse con un nulo legítimo, así que devuelve el
centinela `"__PARSE_ERROR__"`, que luego se cuenta en el informe. **No se descarta el
error en silencio: se cuenta.**

**`strip_html(s)`**
Las sinopsis traen etiquetas HTML. Convierte `<br>` en salto de línea real (que es su
significado), elimina el resto de etiquetas y decodifica las entidades (`&amp;` → `&`).
Devuelve `None` si el resultado queda vacío, para no dejar cadenas de longitud cero
haciéndose pasar por texto.

**`clean_text_series(s)`**
Recorta espacios y convierte los `NULL_SENTINELS` en nulos reales. Es la aplicación
práctica del criterio de la celda 1.1.

**`as_naive_utc(ts)`**
Un `Timestamp` sin zona horaria y otro con zona no se pueden restar. Esta función
normaliza a UTC sin zona, tanto si el valor de entrada la lleva como si no, y evita el
error que aparecía al calcular la antigüedad de las cuentas.


In [ ]:
def explode_list_column(list_array, id_values, value_name, id_name):
    lens = pc.fill_null(pc.list_value_length(list_array), 0).to_numpy(zero_copy_only=False)
    lens = lens.astype(np.int64)
    total = int(lens.sum())
    if total == 0:
        return pd.DataFrame({id_name: [], "pos": [], value_name: []})
    parent = np.repeat(np.asarray(id_values), lens)
    starts = np.concatenate([[0], np.cumsum(lens)[:-1]])
    pos = np.arange(total, dtype=np.int64) - np.repeat(starts, lens)
    values = pc.list_flatten(list_array).to_pandas()
    return pd.DataFrame({id_name: parent, "pos": pos, value_name: values.values})


def parse_json_safe(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    try:
        return json.loads(s)
    except Exception:
        return "__PARSE_ERROR__"


def strip_html(s):
    if not isinstance(s, str):
        return None
    t = re.sub(r"<br\s*/?>", "\n", s)
    t = re.sub(r"<[^>]+>", "", t)
    return html.unescape(t).strip() or None


def clean_text_series(s):
    """Normaliza texto libre: recorta espacios y convierte centinelas literales en nulos."""
    out = s.astype("object").where(s.notna())
    out = out.map(lambda v: v.strip() if isinstance(v, str) else v)
    return out.where(~out.isin(NULL_SENTINELS))


def as_naive_utc(ts):
    """Convierte cualquier timestamp (naive o con zona) a Timestamp naive en UTC."""
    t = pd.Timestamp(ts)
    return t.tz_convert("UTC").tz_localize(None) if t.tz is not None else t


# ETAPA 1: CATÁLOGO DE ANIME (AniList)

**Objetivo.** Construir la tabla base de anime con un identificador MAL estable y
desplegar los atributos anidados que vienen en JSON, para dejar una estructura tabular
sobre la que se pueda unir el resto del pipeline.

**Entrada:** `anilist_anime_data_complete.csv` (438 MB, ~20.400 filas × 62 columnas).
**Salida:** `anime.parquet` más once tablas largas de atributos.

**Criterio de esta etapa**

- `mal_id` es la clave de unión con todo lo demás. Sin ella una fila no puede cruzarse
  con las interacciones, así que las filas sin `idMal` **no se borran**: se apartan a su
  propio fichero para que sigan siendo auditables.
- Los duplicados por `mal_id` se resuelven de forma **determinista y reproducible**, y las
  filas descartadas se guardan aparte.
- Las fechas solo se completan cuando los tres componentes (año, mes, día) son válidos.
  Un año suelto no se convierte en «1 de enero».
- Cada columna JSON se despliega a una tabla larga propia y **el JSON original se conserva**
  en `anime.parquet`: si mañana hace falta un campo que hoy no se extrae, no hay que
  volver a descargar nada.

La etapa se divide en tres funciones (lectura, fechas y despliegue de JSON) más un
orquestador, para poder ejecutar y revisar cada parte por separado.


## 2.1 Lectura del CSV, filas sin `idMal` y duplicados

**Qué hace**

1. Lee el CSV completo con `low_memory=False`, que fuerza a pandas a inferir el tipo de
   cada columna mirando el fichero entero. Con columnas tan heterogéneas, la inferencia
   por bloques produce avisos y tipos mezclados.
2. Separa las filas **sin `idMal`**. Son títulos que existen en AniList pero no tienen
   equivalente en MyAnimeList: no se pueden cruzar con las interacciones, pero tampoco se
   tiran. Van a `anime_anilist_sin_mal.parquet`.
3. Resuelve los **duplicados por `mal_id`**.

**Criterio de desempate.** Varias entradas de AniList pueden apuntar al mismo `mal_id`
(reediciones, versiones alternativas). Se ordena por `mal_id`, luego por `popularity`
descendente y luego por `id` de AniList ascendente; tras esa ordenación, la primera fila
de cada grupo es la que se conserva.

El tercer criterio (`id`) es el que hace el resultado **reproducible**: si dos filas
empatan en popularidad, sin él el ganador dependería del orden en que pandas leyó el CSV,
y dos ejecuciones podrían dar catálogos distintos.

Las filas descartadas se escriben en `anime_duplicados_descartados.parquet` con todas sus
columnas, de modo que la decisión se puede revisar y revertir.


In [ ]:
def cargar_catalogo_anilist(st):
    """Lee el CSV de AniList, aparta las filas sin idMal y resuelve los duplicados.
    
    Devuelve (catalogo_deduplicado, filas_sin_idMal)."""
    al = pd.read_csv(F_ANILIST, low_memory=False)
    st["input_rows"] = len(al)
    st["input_columns"] = al.shape[1]
    log(f"  leidas {len(al):,} filas x {al.shape[1]} columnas")

    no_mal = al[al["idMal"].isna()].copy()
    al = al[al["idMal"].notna()].copy()
    al["mal_id"] = al["idMal"].astype("int64")
    st["rows_without_idMal"] = len(no_mal)
    log(f"  {len(no_mal):,} filas sin idMal -> anime_anilist_sin_mal.parquet")

    # Desempate determinista: mayor popularity y, a igualdad, menor `id` de AniList,
    # para que el resultado no dependa del orden de lectura del CSV. Tras ordenar, la
    # primera fila de cada grupo es la que se conserva y el resto son las descartadas.
    al = al.sort_values(["mal_id", "popularity", "id"], ascending=[True, False, True])
    descartadas = al.duplicated("mal_id", keep="first")
    if descartadas.any():
        dropped = al[descartadas]
        st["duplicate_mal_id_groups"] = int(dropped["mal_id"].nunique())
        st["duplicate_rows_moved_aside"] = int(len(dropped))
        log(
            f"  {dropped['mal_id'].nunique()} mal_id duplicados -> se conserva el de mayor popularity; "
            f"{len(dropped)} filas a anime_duplicados_descartados.parquet"
        )
        write(dropped, "anime_duplicados_descartados.parquet")
        al = al[~descartadas]
    st["unique_mal_id"] = int(al["mal_id"].nunique())

    return al, no_mal


## 2.2 Fechas de emisión y limpieza de la sinopsis

**Fechas.** AniList entrega la fecha de estreno y la de fin repartidas en tres columnas
(`startDate_year`, `startDate_month`, `startDate_day`) y cualquiera de ellas puede faltar.
La regla aplicada es estricta:

> solo se construye una fecha completa cuando los **tres** componentes son válidos.

La máscara `complete` marca esas filas y `pd.to_datetime` recibe los componentes ya
anulados fuera de ella. Una serie con año 1998 y mes 4 pero sin día se queda como nula en
`startdate_full`; los componentes sueltos siguen disponibles en sus columnas originales
para quien quiera usarlos. El motivo: convertir «abril de 1998» en «1 de abril de 1998»
inventaría una precisión que el dato no tiene.

El informe registra cuántas fechas se han podido completar, para que la cobertura sea
visible antes de usarlas en cualquier análisis temporal.

**Sinopsis.** `description` viene con HTML incrustado. `description_text` guarda la
versión limpia **junto a** la original, sin sustituirla.


In [ ]:
def derivar_fechas_y_texto(al, st):
    """Construye las fechas completas de emision y limpia el HTML de la sinopsis."""
    for pref in ["startDate", "endDate"]:
        y, m, d = [pd.to_numeric(al[f"{pref}_{k}"], errors="coerce") for k in ("year", "month", "day")]
        complete = y.notna() & m.notna() & d.notna()
        al[f"{pref.lower()}_full"] = pd.to_datetime(
            dict(year=y.where(complete), month=m.where(complete), day=d.where(complete)),
            errors="coerce",
        )
        st[f"{pref}_complete"] = int(complete.sum())

    al["description_text"] = al["description"].map(strip_html)

    return al


## 2.3 Despliegue de las columnas JSON a tablas largas

Doce columnas de AniList contienen JSON: géneros, sinónimos, tags, estudios, distribución
de notas y de estados, rankings, recomendaciones, relaciones, personajes, staff y enlaces
externos.

**Cómo funciona**

1. `parse_json_safe` convierte cada celda; los fallos de parseo se cuentan por columna en
   `REPORT` en lugar de propagarse.
2. `rows_from(col, fn)` recorre los pares *(mal_id, objeto)* y aplica `fn` a cada elemento
   de la lista, generando una fila por elemento. Recibe también la posición `pos`, que
   algunas tablas conservan porque el orden es informativo (el primer género no es
   equivalente al quinto).
3. Cada resultado se escribe en su propia tabla larga.

**Criterio de las claves.** Las tablas de atributos (géneros, tags, personajes, rankings…)
se unen por `mal_id`. Las tablas de grafo (relaciones, recomendaciones) usan
`mal_id_from` / `mal_id_to`, porque describen una arista entre dos títulos y no un
atributo de uno solo.

**Sobre los accesos defensivos.** Se usa `(s.get("node") or {}).get("id")` en lugar de
`s["node"]["id"]`: la API devuelve `null` en `node` para algunas entradas, y con el acceso
directo una sola fila mal formada interrumpiría el despliegue de las 19.000 restantes.

**Nota de escala.** El despliegue es un bucle de Python sobre ~19.000 filas, no una
operación vectorizada. Es aceptable a este tamaño (unos segundos) y a cambio el criterio
de extracción queda explícito en cada lambda. Sobre las tablas de millones de filas —la
etapa 3— este enfoque no se usa.


In [ ]:
def expandir_columnas_json(al, st):
    """Despliega las doce columnas JSON de AniList a tablas largas."""
    json_cols = [
        "genres", "synonyms", "tags", "studios",
        "stats_scoreDistribution", "stats_statusDistribution",
        "recommendations", "relations", "characters", "staff",
        "rankings", "externalLinks",
    ]
    parsed = {c: al[c].map(parse_json_safe) for c in json_cols if c in al.columns}
    st["json_parse_errors"] = {
        c: int(v.map(lambda x: x == "__PARSE_ERROR__").sum()) for c, v in parsed.items()
    }

    mal = al["mal_id"].values

    def rows_from(col, fn):
        out = []
        if col not in parsed:
            return pd.DataFrame(out)
        for mid, obj in zip(mal, parsed[col]):
            if obj is None or not isinstance(obj, list):
                continue
            for pos, item in enumerate(obj):
                r = fn(mid, pos, item)
                if r:
                    out.extend(r if isinstance(r, list) else [r])
        return pd.DataFrame(out)

    write(rows_from("genres", lambda m, p, g: {"mal_id": m, "pos": p, "genre": g}),
          "anime_generos.parquet")
    write(rows_from("synonyms", lambda m, p, s: {"mal_id": m, "pos": p, "synonym": s}),
          "anime_sinonimos.parquet")
    write(
        rows_from(
            "tags",
            lambda m, p, t: {
                "mal_id": m,
                "tag_id": t.get("id"),
                "tag": t.get("name"),
                "category": t.get("category"),
                "rank": t.get("rank"),
                "is_general_spoiler": t.get("isGeneralSpoiler"),
                "is_media_spoiler": t.get("isMediaSpoiler"),
                "is_adult": t.get("isAdult"),
            },
        ),
        "anime_tags.parquet",
    )
    write(
        rows_from(
            "studios",
            lambda m, p, s: {
                "mal_id": m,
                "studio_id": (s.get("node") or {}).get("id"),
                "studio": (s.get("node") or {}).get("name"),
                "is_main": s.get("isMain"),
                "is_animation_studio": (s.get("node") or {}).get("isAnimationStudio"),
            },
        ),
        "anime_estudios.parquet",
    )
    write(
        rows_from(
            "stats_scoreDistribution",
            lambda m, p, s: {"mal_id": m, "score_anilist": s.get("score"), "amount": s.get("amount")},
        ),
        "anime_distribucion_score.parquet",
    )
    write(
        rows_from(
            "stats_statusDistribution",
            lambda m, p, s: {"mal_id": m, "status": s.get("status"), "amount": s.get("amount")},
        ),
        "anime_distribucion_status.parquet",
    )
    write(
        rows_from(
            "rankings",
            lambda m, p, r: {
                "mal_id": m,
                "rank": r.get("rank"),
                "type": r.get("type"),
                "format": r.get("format"),
                "year": r.get("year"),
                "season": r.get("season"),
                "all_time": r.get("allTime"),
                "context": r.get("context"),
            },
        ),
        "anime_rankings.parquet",
    )

    def rec_row(m, p, r):
        node = r.get("node") or {}
        media = node.get("mediaRecommendation") or {}
        return {
            "mal_id_from": m,
            "rating": node.get("rating"),
            "anilist_id_to": media.get("id"),
            "mal_id_to": media.get("idMal"),
        }
    write(rows_from("recommendations", rec_row), "anime_recomendaciones.parquet")

    def rel_row(m, p, r):
        node = r.get("node") or {}
        return {
            "mal_id_from": m,
            "relation_type": r.get("relationType"),
            "anilist_id_to": node.get("id"),
            "mal_id_to": node.get("idMal"),
            "type_to": node.get("type"),
            "format_to": node.get("format"),
        }
    write(rows_from("relations", rel_row), "anime_relaciones.parquet")

    def char_row(m, p, c):
        node = c.get("node") or {}
        name = node.get("name") or {}
        return {
            "mal_id": m,
            "anilist_character_id": node.get("id"),
            "role": c.get("role"),
            "name_full": name.get("full"),
            "name_native": name.get("native"),
            "favourites": node.get("favourites"),
        }
    write(rows_from("characters", char_row), "anime_personajes_anilist.parquet")

    def staff_row(m, p, s):
        node = s.get("node") or {}
        name = node.get("name") or {}
        return {
            "mal_id": m,
            "anilist_staff_id": node.get("id"),
            "role": s.get("role"),
            "name_full": name.get("full"),
            "name_native": name.get("native"),
        }
    write(rows_from("staff", staff_row), "anime_staff.parquet")


## 2.4 Orquestador de la etapa 1

Encadena las tres funciones anteriores y escribe las dos salidas finales:

- `anime_anilist_sin_mal.parquet`: las filas apartadas por no tener `idMal`.
- `anime.parquet`: el catálogo limpio, con `mal_id` e `id` en cabecera.

En la selección final de columnas se descarta `idMal`, ya normalizada como `mal_id`,
pero **se conservan todas las demás columnas originales**, incluidas las que contienen
JSON en crudo. El catálogo pesa unos 70 MB: mantener el original completo es barato
comparado con tener que volver a descargar la fuente.


In [ ]:
def stage1_anime():
    """Orquestador de la etapa 1."""
    log("ETAPA 1: catalogo de anime desde AniList")
    st = REPORT["stages"]["1_anime"] = {}

    al, no_mal = cargar_catalogo_anilist(st)
    al = derivar_fechas_y_texto(al, st)
    expandir_columnas_json(al, st)

    write(no_mal, "anime_anilist_sin_mal.parquet")

    cols = ["mal_id", "id"] + [c for c in al.columns if c not in ("mal_id", "id", "idMal")]
    write(al[cols], "anime.parquet")
    log("ETAPA 1 completada\n")


# ETAPA 2: PERFILES, FAVORITOS Y AMISTADES

**Objetivo.** Construir el perfil de usuario con validaciones verificables, conservando
siempre el dato original, y desplegar las relaciones sociales y los favoritos a tablas
largas sin perder el orden.

**Entrada:** `profile.parquet` (1,9 GB, 18,2 millones de perfiles) y
`ulist_anime_fetch.parquet` (el registro de la descarga).
**Salida:** `users.parquet`, cinco tablas de favoritos, `user_amigos.parquet`,
`user_insignias.parquet`, `user_texto.parquet` y la cohorte de trabajo.

**Criterio de esta etapa**

- **La fecha del snapshot no se escribe a mano**: se deduce del `modified_at` máximo del
  registro de descarga. Es el único momento verificable al que referir la antigüedad de
  las cuentas.
- **Los valores imposibles se marcan, no se corrigen.** Una media de 99 sobre 10 queda tal
  cual en el dato, con una bandera booleana al lado. Quien analice decide si la excluye.
- **Una bandera nula no es una bandera falsa.** Si no hay dato con el que comprobar algo,
  la bandera queda nula: `False` significaría «comprobado y correcto», que es una
  afirmación distinta de «no se ha podido comprobar».
- **El texto libre se separa** de la tabla principal, porque infla el fichero y casi nunca
  se necesita junto al resto de atributos.

La etapa se divide en cinco funciones más un orquestador.


## 3.1 Registro de descarga: fecha del snapshot y listas completas

El fichero `ulist_anime_fetch.parquet` es el log del proceso de scraping: una fila por
usuario con el código HTTP obtenido al intentar descargar su lista.

**Dos cosas se extraen de aquí:**

1. **La fecha del snapshot**, como el `modified_at` máximo. Todo el dataset es una foto
   de un momento concreto, y esa fecha es la referencia contra la que se calcula la
   antigüedad de las cuentas. Deducirla del log en vez de escribirla a mano hace que el
   pipeline siga siendo correcto si mañana se vuelve a descargar la fuente.

2. **Qué usuarios tienen su lista efectivamente descargada** (código HTTP 200). Es una
   distinción importante: que un perfil exista no implica que su lista de anime esté en
   el dataset. Un perfil sin lista descargada tiene cero interacciones **por un fallo de
   descarga**, no porque el usuario no vea anime. Confundir ambas cosas sesga cualquier
   cálculo de actividad.

El reparto completo de códigos HTTP queda en el informe. La conversión a índice único
protege frente a usuarios repetidos en el log.


In [ ]:
def leer_log_de_descarga(st):
    """Fecha del snapshot y usuarios cuya lista se descargo correctamente.
    
    Devuelve (snapshot, snapshot_sin_zona, indice_de_usuarios_con_lista)."""
    fetch = pq.read_table(F_FETCH, columns=["username", "status", "modified_at"])
    snapshot = pc.max(fetch["modified_at"]).as_py()
    snapshot_naive = as_naive_utc(snapshot)
    st["snapshot_utc"] = str(snapshot)
    log(f"  fecha del snapshot (max modified_at del fetch log): {snapshot}")

    fs = fetch["status"].combine_chunks()
    st["fetch_status_counts"] = {str(d["values"]): d["counts"] for d in pc.value_counts(fs).to_pylist()}
    fetch_ok = pc.filter(fetch["username"].combine_chunks(), pc.equal(fs, 200)).to_pandas()
    st["users_with_list_fetched_ok"] = int(len(fetch_ok))
    fetch_ok_set = pd.Index(fetch_ok.values).unique()

    return snapshot, snapshot_naive, fetch_ok_set


## 3.2 Fechas del perfil: alta, última conexión y cumpleaños

Las tres columnas de fecha vienen como texto en formato de MyAnimeList y se parsean con
`errors="coerce"`: lo que no encaje con el formato queda nulo, y el informe registra
cuántos valores se han podido interpretar.

| Columna origen | Formato | Destino |
|---|---|---|
| `joined` | `Nov 5, 2014` | `joined_date` |
| `last_online` | `Jul 2, 2025 9:16 PM` | `last_online_ts` |
| `birthday` | dos formatos posibles | `birth_date`, `birth_year`, `birth_month`, `birth_day` |

**El caso del cumpleaños.** MyAnimeList permite ocultar el año de nacimiento, así que el
campo llega de dos maneras: `Mar 23, 1990` o solo `Mar 23`. Se distinguen con dos
expresiones regulares y se tratan por separado:

- Con año: se construye la fecha completa y se rellenan las cuatro columnas.
- Sin año: se parsea añadiendo el año ficticio 1904 **solo para poder extraer mes y día**.
  Ese año nunca llega al resultado: `birth_year` queda nulo y solo se conservan
  `birth_month` y `birth_day`.

Se usa 1904 por ser bisiesto, para que un `29 de febrero` no se pierda al parsear.

Así, quien calcule edades usa `birth_year` y automáticamente excluye a quien no lo
publicó, mientras que un análisis de estacionalidad de cumpleaños puede aprovechar los
187.000 perfiles que sí dieron mes y día. **Los tipos son enteros con soporte de nulos**
(`Int16`, `Int8`) precisamente para que la ausencia se pueda representar.


In [ ]:
def normalizar_fechas_de_perfil(df, st):
    """Parsea alta, ultima conexion y cumpleanos. Modifica `df` en el sitio."""
    df["joined_date"] = pd.to_datetime(df["joined"], format="%b %d, %Y", errors="coerce")
    st["joined_parsed"] = int(df["joined_date"].notna().sum())
    df["last_online_ts"] = pd.to_datetime(
        df["last_online"], format="%b %d, %Y %I:%M %p", errors="coerce"
    )
    st["last_online_parsed"] = int(df["last_online_ts"].notna().sum())

    bd = df["birthday"].fillna("").str.strip()
    with_year = bd.str.match(r"^[A-Za-z]{3} \d{1,2}, \d{4}$")
    no_year = bd.str.match(r"^[A-Za-z]{3} \d{1,2}$")
    full = pd.to_datetime(bd.where(with_year), format="%b %d, %Y", errors="coerce")
    part = pd.to_datetime(bd.where(no_year) + ", 1904", format="%b %d, %Y", errors="coerce")
    df["birth_date"] = full
    df["birth_year"] = full.dt.year.astype("Int16")
    df["birth_month"] = full.dt.month.fillna(part.dt.month).astype("Int8")
    df["birth_day"] = full.dt.day.fillna(part.dt.day).astype("Int8")
    st["birthday_nonempty"] = int((bd != "").sum())
    st["birthday_with_year"] = int(with_year.sum())
    st["birthday_month_day_only"] = int(no_year.sum())
    log(f"  birthday: {int((bd != '').sum()):,} no vacios, {int(with_year.sum()):,} con ano")


## 3.3 Validaciones: marcar lo imposible sin tocar el dato

Aquí se aplica la regla central de esta etapa: **lo imposible se marca, no se corrige**.
Ninguna de estas comprobaciones modifica el valor original; todas añaden una columna
booleana al lado.

**1. Medias fuera de escala** (`anime_mean_invalid`, `manga_mean_invalid`).
La escala de MyAnimeList es 1-10, y el dataset contiene alguna media por encima de 10.
La bandera es de tipo `boolean` (booleano con nulos), no `bool`:

- `True` — comprobado, el valor es imposible.
- `False` — comprobado, el valor está en escala.
- nulo — no hay media, no se ha podido comprobar.

La distinción no es cosmética: la versión anterior calculaba `valor > 10`, y en pandas
`NaN > 10` da `False`, con lo que todos los perfiles sin media quedaban marcados como
«válidos».

**2. Medias en cero.** Se cuentan aparte en el informe (`*_zero_means_unrated`). La fuente
escribe `0.00` cuando el usuario **no ha puntuado nada**, no cuando puntúa con un cero.
Son 10,4 millones de perfiles. Aquí no se altera el valor —el criterio de la etapa es no
tocar el dato—, pero queda registrado para que el bloque analítico lo excluya.

**3. Días de anime imposibles** (`anime_days_invalid`). `anime_days` es el tiempo total
declarado de visionado. Si supera la antigüedad de la cuenta, el dato es imposible. La
comprobación exige que existan **los dos** valores: sin fecha de alta o sin `anime_days`
la bandera queda nula, no `False`.

**4. Ubicación.** Se pasa por `clean_text_series` para convertir centinelas en nulos.

**5. Género.** El código entero se conserva **sin traducir a etiquetas**: la fuente no
publica el diccionario de códigos. Lo que sí se hace es dejar el reparto de códigos en el
informe, para que quien quiera interpretarlos parta de evidencia.

**6. Lista descargada** (`list_fetched_ok`). Cruza el perfil con el registro de la celda
anterior. Es la bandera que permite distinguir «no vio nada» de «no se pudo descargar».


In [ ]:
def marcar_valores_imposibles(df, snapshot_naive, fetch_ok_set, st):
    """Anade las banderas de validacion sin alterar los valores originales."""
    # Las banderas de invalidez son booleanas con nulos: si la media no existe no se
    # puede afirmar que sea valida (antes NaN > 10 daba False y se leia como "valida").
    for c in ["anime_mean", "manga_mean"]:
        flag = (df[c] > SCORE_MAX).astype("boolean")
        df[f"{c}_invalid"] = flag.where(df[c].notna())
        st[f"{c}_out_of_scale"] = int(flag.fillna(False).sum())
        # La fuente escribe 0.00 cuando el usuario no ha puntuado nada: no es una
        # nota real. Se conserva el valor original y se deja constancia del volumen.
        st[f"{c}_zero_means_unrated"] = int((df[c] == 0).sum())

    account_days = (snapshot_naive - df["joined_date"]).dt.days
    df["account_age_days"] = account_days.astype("Int32")
    # La comprobacion solo tiene sentido si existen los dos valores: si falta alguno
    # la bandera queda nula, no False (False se leeria como "verificado y correcto").
    comprobable = account_days.notna() & df["anime_days"].notna()
    imposible = (df["anime_days"] > account_days) & comprobable
    df["anime_days_invalid"] = imposible.astype("boolean").where(comprobable)
    st["anime_days_exceeding_account_age"] = int(imposible.sum())
    st["anime_days_not_checkable"] = int((~comprobable).sum())
    log(
        f"  anime_mean fuera de escala: {st['anime_mean_out_of_scale']:,} | "
        f"anime_days > edad de cuenta: {int(imposible.sum()):,}"
    )

    df["location"] = clean_text_series(df["location"])
    st["location_nonempty"] = int(df["location"].notna().sum())
    # `gender` viene como codigo entero sin diccionario publicado por la fuente:
    # se conserva el codigo crudo y se deja el conteo en el informe, sin inventar etiquetas.
    st["gender_code_counts"] = {str(k): int(v) for k, v in df["gender"].value_counts(dropna=False).items()}

    df["uid_missing"] = df["uid"].isna()
    st["uid_missing"] = int(df["uid_missing"].sum())
    df["list_fetched_ok"] = df["username"].isin(fetch_ok_set)
    st["profiles_with_list_fetched_ok"] = int(df["list_fetched_ok"].sum())


## 3.4 Despliegue de favoritos, amigos e insignias

Siete columnas del perfil son listas: cinco de favoritos, una de amigos y una de
insignias. Cada una se despliega a su propia tabla larga con `explode_list_column`,
que **preserva la posición original** en la columna `pos`.

La posición es información real: en una lista de favoritos, el primer elemento es la
elección principal del usuario. Aplanar sin conservar el orden la perdería.

**Advertencia sobre el grafo social.** El máximo observado de amigos por usuario es
exactamente 10, lo que significa que **la lista viene truncada en origen**: la fuente
devuelve como mucho diez amistades por perfil. `user_amigos.parquet` es por tanto un
**grafo social parcial**, y cualquier métrica de red calculada sobre él (grado, centralidad,
componentes conexas) estará sesgada por ese tope. La advertencia se guarda en el propio
informe para que viaje con los datos y no se pierda en una conversación.


In [ ]:
def exportar_listas_de_usuario(t, df, st):
    """Despliega favoritos, amigos e insignias a tablas largas con su posicion."""
    usernames = df["username"].values
    fav_counts = {}
    for col, out in [
        ("fav_anime", "user_favoritos_anime.parquet"),
        ("fav_manga", "user_favoritos_manga.parquet"),
        ("fav_character", "user_favoritos_personaje.parquet"),
        ("fav_people", "user_favoritos_persona.parquet"),
        ("fav_company", "user_favoritos_empresa.parquet"),
    ]:
        ex = explode_list_column(t[col].combine_chunks(), usernames, "item_id", "username")
        fav_counts[col] = len(ex)
        write(ex, out)
    st["favorite_rows"] = fav_counts

    fr = explode_list_column(t["friends"].combine_chunks(), usernames, "friend_username", "username")
    st["friend_edges"] = len(fr)
    st["friends_note"] = (
        "El maximo observado de amigos por usuario es 10: la lista viene truncada en origen. "
        "Es un grafo social PARCIAL."
    )
    write(fr, "user_amigos.parquet")

    bg = explode_list_column(t["badges"].combine_chunks(), usernames, "badge", "username")
    write(bg, "user_insignias.parquet")


## 3.5 Texto libre: `about` y `comm`

Las dos columnas de texto libre se separan a `user_texto.parquet`. Salen de la tabla
principal por tamaño: pesan más que todo el resto del perfil junto y rara vez se necesitan
en el mismo análisis.

**El problema de los centinelas.** El filtro obvio —descartar la cadena vacía— no
funciona con esta fuente. Comprobado sobre `profile.parquet`:

- `comm` contiene la cadena literal `"null"` en **1.028.505 de 1.048.576** filas del
  primer bloque (98%).
- Con el filtro por cadena vacía, `comm` nunca era nulo, así que **todos** los usuarios
  pasaban el filtro `about no nulo O comm no nulo`.

Resultado: `user_texto.parquet` guardaba los 18,2 millones de perfiles y ocupaba 1,2 GB,
casi todo con la palabra `null` repetida. Pasando ambas columnas por `clean_text_series`
solo quedan los perfiles con texto real, y el informe anota cuántos hay de cada tipo.


In [ ]:
def exportar_texto_libre(df, st):
    """Separa `about` y `comm` a su propia tabla, filtrando los centinelas de la fuente."""
    # `about` y `comm` traen centinelas de texto: `comm` vale la cadena "null" en ~98% de
    # las filas. Filtrar solo por cadena vacia dejaba pasar los 18.2M de perfiles.
    txt = df[["username", "about", "comm"]].copy()
    for c in ["about", "comm"]:
        txt[c] = clean_text_series(txt[c])
    st["about_nonempty"] = int(txt["about"].notna().sum())
    st["comm_nonempty"] = int(txt["comm"].notna().sum())
    txt = txt[txt["about"].notna() | txt["comm"].notna()]
    st["users_with_free_text"] = int(len(txt))
    log(f"  texto libre real: {len(txt):,} usuarios de {len(df):,}")
    write(txt, "user_texto.parquet")


## 3.6 Orquestador de la etapa 2 y cohorte de trabajo

Encadena las cinco funciones anteriores y produce las dos salidas finales.

**`users.parquet`** — el perfil completo sin las columnas de texto libre, más las columnas
derivadas: fechas parseadas, banderas de validación, antigüedad de la cuenta y
`fav_anime_n` (número de favoritos, calculado sin desplegar la lista).

**`cohorte_favoritos.parquet`** — la cohorte de trabajo para la etapa 3, definida por dos
condiciones simultáneas:

1. `fav_anime_n > 0` — el usuario ha marcado al menos un anime como favorito.
2. `list_fetched_ok` — su lista se descargó correctamente.

**Por qué existe la cohorte.** El fichero de interacciones tiene 968 millones de filas.
Filtrar por usuarios con señal de preferencia explícita lo reduce a 527 millones sin
perder los perfiles informativos: quien no ha marcado ningún favorito aporta muy poco a un
análisis de preferencias y sí mucho ruido y volumen.

Es una decisión de alcance, no de limpieza, y por eso queda registrada en el informe con
su tamaño exacto. La etapa 3 admite otras cohortes (`fetched`, `all`) sin tocar el código.


In [ ]:
def stage2_users():
    """Orquestador de la etapa 2."""
    log("ETAPA 2: perfiles de usuario")
    st = REPORT["stages"]["2_users"] = {}

    snapshot, snapshot_naive, fetch_ok_set = leer_log_de_descarga(st)

    t = pq.read_table(F_PROFILE)
    n = t.num_rows
    st["input_rows"] = n
    log(f"  perfiles leidos: {n:,}")

    scalar_cols = [f.name for f in t.schema if not pa.types.is_list(f.type)]
    df = t.select(scalar_cols).to_pandas()

    normalizar_fechas_de_perfil(df, st)
    marcar_valores_imposibles(df, snapshot_naive, fetch_ok_set, st)
    exportar_listas_de_usuario(t, df, st)
    exportar_texto_libre(df, st)

    lens = pc.fill_null(pc.list_value_length(t["fav_anime"].combine_chunks()), 0).to_numpy(zero_copy_only=False)
    df["fav_anime_n"] = lens.astype("int32")

    users = df.drop(columns=["about", "comm"])
    write(users, "users.parquet")

    cohort = users.loc[(users["fav_anime_n"] > 0) & users["list_fetched_ok"], "username"]
    st["cohort_favorites_size"] = int(len(cohort))
    write(cohort.to_frame("username"), "cohorte_favoritos.parquet")
    log(f"  cohorte (fav_anime no vacio + lista OK): {len(cohort):,} usuarios")
    log("ETAPA 2 completada\n")


# ETAPA 3: INTERACCIONES Y AGREGADOS POR ANIME

**Objetivo.** Procesar la tabla de hechos —la relación usuario × anime— y construir los
agregados por título.

**Entrada:** `ulist_anime.parquet`, **12,5 GB y 968 millones de filas**.
**Salida:** `interacciones.parquet` (la cohorte, ~527 millones de filas) y
`anime_stats_globales.parquet` (un resumen por anime).

**El problema de escala y cómo se resuelve**

Esta tabla no cabe en memoria. Todo el diseño de la etapa se deriva de ahí:

- Se recorre **row group a row group** (924 bloques) y se escribe con un
  `ParquetWriter` abierto, sin acumular nada en memoria.
- Los agregados se calculan **al vuelo con `np.bincount`** sobre arrays indexados por
  `anime_id`. Un `groupby` de pandas sobre 968 millones de filas no es viable; `bincount`
  recorre el bloque una vez y suma sobre un array preasignado.
- Los contadores de calidad se acumulan en un diccionario y acaban en el informe.

**Diferencia de alcance importante**

| Salida | Sobre qué se calcula |
|---|---|
| `interacciones.parquet` | **Solo la cohorte** seleccionada en la etapa 2 |
| `anime_stats_globales.parquet` | **Todas** las filas leídas del origen |

Las dos tablas **no son comparables fila a fila**: los agregados describen a toda la
población de MyAnimeList y las interacciones solo a la cohorte. Es deliberado —así los
agregados por anime no quedan sesgados por el filtro— y queda anotado en el informe.

La etapa se divide en seis funciones más un orquestador.


## 4.1 Tamaño de los acumuladores y carga de la cohorte

**`_max_anime_id(pf)`** — obtiene el `anime_id` máximo **leyendo solo las estadísticas de
los metadatos del parquet**, sin tocar los datos. Cada row group guarda el mínimo y el
máximo de cada columna; recorrer 924 bloques de metadatos es instantáneo frente a leer
12,5 GB.

Ese máximo dimensiona los arrays de acumulación. La versión anterior usaba un tope fijo
de 70.000 y **descartaba en silencio** cualquier fila con un `anime_id` mayor. Hoy no se
supera, pero MyAnimeList sigue asignando identificadores: era un fallo latente que
aparecería sin aviso al actualizar la fuente. Si el parquet no trae estadísticas, se cae a
un tope amplio en lugar de fallar.

**`_cargar_cohorte(cohort_mode, st)`** — devuelve el conjunto de usuarios a conservar:

| Modo | Origen | Uso |
|---|---|---|
| `favorites` | `cohorte_favoritos.parquet` | Por defecto: usuarios con favoritos y lista descargada |
| `fetched` | `users.parquet` filtrado por `list_fetched_ok` | Todos los que tienen lista |
| `all` | — | Sin filtro |

Los nombres se pasan por `.unique()`. La versión anterior usaba `Index.get_indexer`, que
**exige un índice sin repeticiones** y lanza `InvalidIndexError` con un solo usuario
duplicado en la cohorte.


In [ ]:
def _max_anime_id(pf):
    """Maximo anime_id del fichero leido de las estadisticas parquet (sin recorrer datos)."""
    names = [pf.metadata.row_group(0).column(i).path_in_schema for i in range(pf.metadata.num_columns)]
    if "anime_id" not in names:
        return None
    ci = names.index("anime_id")
    mx = None
    for i in range(pf.metadata.num_row_groups):
        stt = pf.metadata.row_group(i).column(ci).statistics
        if stt is None or not stt.has_min_max:
            return None
        mx = int(stt.max) if mx is None else max(mx, int(stt.max))
    return mx


def _cargar_cohorte(cohort_mode, st):
    """Conjunto de usuarios a conservar segun el modo de cohorte.
    
    Devuelve None cuando no hay que filtrar (modo 'all')."""
    if cohort_mode == "all":
        return None
    if cohort_mode == "favorites":
        src = os.path.join(OUT, "cohorte_favoritos.parquet")
    elif cohort_mode == "fetched":
        src = os.path.join(OUT, "users.parquet")
    else:
        raise ValueError(f"cohorte desconocida: {cohort_mode}")
    if not os.path.exists(src):
        raise FileNotFoundError(f"Falta {src}. Ejecuta primero la etapa 2.")
    cu = pq.read_table(
        src,
        columns=["username"] if cohort_mode == "favorites" else ["username", "list_fetched_ok"],
    ).to_pandas()
    if cohort_mode == "fetched":
        cu = cu[cu["list_fetched_ok"]]
    # `unique()` evita que un username repetido rompa la busqueda de pertenencia:
    # `Index.get_indexer` exige un indice sin repeticiones y lanzaba InvalidIndexError.
    cohort_idx = pd.Index(cu["username"].values).unique()
    st["cohort_size"] = int(len(cohort_idx))
    log(f"  cohorte: {len(cohort_idx):,} usuarios")
    return cohort_idx


## 4.2 Esquema de salida y contadores de calidad

**El esquema se declara explícitamente** en lugar de dejar que pyarrow lo infiera de cada
bloque. Con 924 escrituras sucesivas sobre el mismo fichero, un bloque en el que una
columna llegue entera de nulos podría inferir un tipo distinto y romper la escritura a
mitad del proceso. Declarándolo se fijan además los tipos ajustados: `int8` para estado y
puntuación, `date32` para fechas sin hora, `int32` para `anime_id`.

**Columnas conservadas en pares.** `start_date` / `start_date_raw` y su equivalente de
fin: la primera es la fecha parseada, la segunda el texto original. Se guardan las dos
porque un 12% de las fechas no se puede interpretar, y con el texto original a la vista se
puede diagnosticar por qué en lugar de tener solo un nulo.

**Los contadores** registran todo lo que se descarta o resulta sospechoso:

| Contador | Qué mide |
|---|---|
| `rows_seen` / `rows_written` | Leídas del origen frente a escritas tras el filtro de cohorte |
| `username_not_in_cohort` | Filas descartadas por el filtro |
| `status_null` / `status_unknown_code` | Estado ausente frente a estado con un código no documentado |
| `score_missing` / `score_out_of_range` | Puntuación ausente (nulo o cero) frente a fuera de la escala 1-10 |
| `created_at_zero` / `updated_at_zero` | Marcas de tiempo a cero, que son nulos codificados |
| `start_date_unparsed` / `finish_date_unparsed` | Texto de fecha no interpretable |
| `start_after_updated` | Inicio posterior a la última modificación: cronología imposible |
| `anime_id_out_of_range` | Identificadores fuera del rango de los acumuladores |

`score_missing` y `score_out_of_range` estaban antes fundidos en un único contador
`score_zero` que sumaba 428 millones, mezclando «no puntuado» con «valor inválido». Son
dos hechos distintos y ahora se cuentan por separado.


In [ ]:
# Columnas que se leen de cada row group del fichero de origen.
COLS_ULIST = [
    "username", "anime_id", "status", "score", "created_at", "updated_at",
    "num_watched_episodes", "start_date_string", "finish_date_string",
    "is_rewatching", "priority_string", "tags", "notes", "storage_string",
]

NSTAT = MAX_STATUS_CODE + 1
NSCORE = SCORE_MAX + 1

# El esquema de salida se declara: con 924 escrituras sucesivas, dejar que pyarrow
# lo infiera bloque a bloque puede romper el fichero a mitad de proceso.
ESQUEMA_INTERACCIONES = pa.schema([
    ("username", pa.string()),
    ("anime_id", pa.int32()),
    ("status", pa.int8()),
    ("status_label", pa.string()),
    ("score", pa.int8()),
    ("created_at", pa.timestamp("s", tz="UTC")),
    ("updated_at", pa.timestamp("s", tz="UTC")),
    ("num_watched_episodes", pa.int32()),
    ("start_date", pa.date32()),
    ("finish_date", pa.date32()),
    ("start_date_raw", pa.string()),
    ("finish_date_raw", pa.string()),
    ("start_after_updated", pa.bool_()),
    ("is_rewatching", pa.bool_()),
    ("priority", pa.string()),
    ("tags", pa.string()),
    ("notes", pa.string()),
    ("storage", pa.string()),
])


def _nuevos_contadores():
    """Contadores de calidad de la etapa 3, todos a cero."""
    return dict(
        rows_seen=0, rows_written=0, username_not_in_cohort=0,
        status_null=0, status_unknown_code=0,
        score_missing=0, score_out_of_range=0,
        created_at_zero=0, updated_at_zero=0,
        start_date_unparsed=0, finish_date_unparsed=0,
        start_after_updated=0, anime_id_out_of_range=0,
    )


## 4.3 Acumulación de estadísticas por bloque

El núcleo de rendimiento de la etapa. Se llama una vez por row group y actualiza siete
acumuladores en su sitio, sin construir ningún objeto intermedio.

**Cómo funciona `np.bincount`.** Los acumuladores son arrays de numpy indexados
directamente por `anime_id`: la posición 1535 corresponde al anime 1535.
`np.bincount(a, minlength=MAXID)` cuenta las apariciones de cada identificador del bloque
en una sola pasada en C, y el resultado se suma al acumulador global.

**El truco del índice aplanado.** Para las tablas de dos dimensiones (estado × anime,
puntuación × anime) se aplana el índice: `anime_id * NSTAT + status`. Así una única
llamada a `bincount` resuelve la tabla de contingencia completa. La alternativa,
`np.add.at`, es correcta pero recorre elemento a elemento en Python y sobre 968 millones
de filas la diferencia es de horas.

**Decisiones de tratamiento en esta función**

- Los nulos se rellenan con `-1` (identificador, estado) o se detectan por separado, para
  que nunca se confundan con un valor válido al indexar.
- **`is_rewatching` se acepta como texto `"0"`/`"1"`, como booleano y como entero.** En
  esta fuente llega como texto, pero la comparación directa con `"1"` fallaría en silencio
  —dando cero rewatches— si un volcado futuro lo entrega como booleano.
- Solo entran en los agregados las filas con `anime_id` en rango y, para las notas, las
  que están dentro de la escala 1-10.

La función devuelve la máscara `ok_id`, que el orquestador combina con el filtro de
cohorte.


In [ ]:
def _crear_acumuladores(maxid):
    """Arrays indexados por anime_id sobre los que se suma cada row group."""
    return {
        "maxid": maxid,
        "total": np.zeros(maxid, dtype=np.int64),
        "status": np.zeros(maxid * NSTAT, dtype=np.int64),
        "scored": np.zeros(maxid, dtype=np.int64),
        "scoresum": np.zeros(maxid, dtype=np.int64),
        "hist": np.zeros(maxid * NSCORE, dtype=np.int64),
        "episodes": np.zeros(maxid, dtype=np.int64),
        "rewatch": np.zeros(maxid, dtype=np.int64),
    }


def _acumular_bloque(acc, tb, counters):
    """Suma un row group a los acumuladores globales.

    Trabaja sobre TODAS las filas del bloque, tambien las que luego no pasen el
    filtro de cohorte. Devuelve la mascara de filas con anime_id dentro de rango.
    """
    maxid = acc["maxid"]
    aid_arr = tb["anime_id"].combine_chunks()
    sta_arr = tb["status"].combine_chunks()
    sco_arr = tb["score"].combine_chunks()

    # Los nulos se rellenan con -1 para que nunca se confundan con un valor valido.
    aid = pc.fill_null(aid_arr, -1).to_numpy(zero_copy_only=False).astype(np.int64)
    sta = pc.fill_null(sta_arr, -1).to_numpy(zero_copy_only=False).astype(np.int64)
    sco = pc.fill_null(sco_arr, -1).to_numpy(zero_copy_only=False).astype(np.int64)
    eps = pc.fill_null(tb["num_watched_episodes"].combine_chunks(), 0).to_numpy(zero_copy_only=False)
    rew = tb["is_rewatching"].combine_chunks().to_pandas()

    counters["status_null"] += int(sta_arr.null_count)
    counters["score_missing"] += int(sco_arr.null_count) + int((sco == 0).sum())

    ok_id = (aid >= 0) & (aid < maxid)
    counters["anime_id_out_of_range"] += int((~ok_id).sum())

    a = aid[ok_id]
    acc["total"] += np.bincount(a, minlength=maxid)

    # Tabla de contingencia anime x estado con un unico bincount sobre el indice
    # aplanado `anime_id * NSTAT + status`; np.add.at seria correcto pero recorre
    # elemento a elemento y sobre 968M de filas cuesta horas.
    s_ok = sta[ok_id]
    valid_s = (s_ok >= 0) & (s_ok <= MAX_STATUS_CODE)
    acc["status"] += np.bincount(a[valid_s] * NSTAT + s_ok[valid_s], minlength=maxid * NSTAT)
    counters["status_unknown_code"] += int((~np.isin(s_ok, list(STATUS_LABELS))).sum())

    sc_ok = sco[ok_id]
    m_sc = (sc_ok >= SCORE_MIN) & (sc_ok <= SCORE_MAX)
    counters["score_out_of_range"] += int((sc_ok > SCORE_MAX).sum())
    acc["scored"] += np.bincount(a[m_sc], minlength=maxid)
    acc["scoresum"] += np.bincount(a[m_sc], weights=sc_ok[m_sc], minlength=maxid).astype(np.int64)
    acc["hist"] += np.bincount(a[m_sc] * NSCORE + sc_ok[m_sc], minlength=maxid * NSCORE)
    acc["episodes"] += np.bincount(a, weights=eps[ok_id], minlength=maxid).astype(np.int64)

    # `is_rewatching` llega como texto "0"/"1" en esta fuente, pero se acepta tambien
    # booleano o numerico para no depender del tipo concreto del volcado.
    rew_bool = rew.map({"1": True, "0": False, 1: True, 0: False, True: True, False: False})
    acc["rewatch"] += np.bincount(a[rew_bool.fillna(False).to_numpy(dtype=bool)[ok_id]], minlength=maxid)

    return ok_id


## 4.4 Transformación del bloque que se escribe

Mientras la función anterior trabaja sobre **todas** las filas del bloque para los
agregados, esta se aplica **solo a las filas de la cohorte** que van al fichero de salida.

**Transformaciones aplicadas**

1. **Puntuación**: fuera de la escala 1-10 pasa a nulo, con tipo `Int8` (entero con
   nulos). El cero de origen significa «sin puntuar» y por eso se convierte en nulo aquí:
   dejarlo como cero contaminaría cualquier media.
2. **Estado**: se añade `status_label` legible junto al código numérico. Los códigos no
   documentados quedan con etiqueta nula, no se inventa una.
3. **Marcas de tiempo**: `created_at` y `updated_at` son epoch en segundos, y el valor
   `0` es un nulo codificado (1 de enero de 1970 no es una fecha real de alta aquí). Se
   convierten a nulo y se cuentan.
4. **Fechas de inicio y fin**: texto en formato `%m-%d-%y`. Lo que no se pueda interpretar
   queda nulo y se cuenta; el texto original se conserva en las columnas `_raw`.
5. **Cronología** (`start_after_updated`): marca las filas cuya fecha de inicio es
   posterior a la última modificación del registro, que es imposible. **Se marca, no se
   corrige**, siguiendo el criterio del pipeline. Afecta a unos 3,5 millones de filas.
6. **Texto**: prioridad, tags, notas y almacenamiento pasan por `clean_text_series`.

El bloque se convierte a Arrow **con el esquema declarado**, lo que verifica en cada
escritura que los tipos siguen siendo los esperados.


In [ ]:
def _transformar_bloque(tb, keep, counters):
    """Convierte las filas seleccionadas de un row group en la tabla que se escribe.
    
    A diferencia de _acumular_bloque, solo ve las filas de la cohorte."""
    d = tb.filter(pa.array(keep)).to_pandas()

    d["score"] = d["score"].where(d["score"].between(SCORE_MIN, SCORE_MAX)).astype("Int8")
    d["status_label"] = d["status"].map(STATUS_LABELS)
    d["status"] = d["status"].astype("Int8")

    for c in ["created_at", "updated_at"]:
        z = d[c] == 0
        counters[f"{c}_zero"] += int(z.sum())
        d[c] = pd.to_datetime(d[c].where(~z), unit="s", utc=True)

    for c, out in [("start_date_string", "start_date"), ("finish_date_string", "finish_date")]:
        raw = clean_text_series(d[c])
        d[out] = pd.to_datetime(raw, format="%m-%d-%y", errors="coerce")
        counters[f"{out.split('_')[0]}_date_unparsed"] += int(raw.notna().sum() - d[out].notna().sum())

    bad = d["start_date"].dt.tz_localize("UTC") > d["updated_at"]
    d["start_after_updated"] = bad.astype("boolean")
    counters["start_after_updated"] += int(bad.fillna(False).sum())

    d["is_rewatching"] = d["is_rewatching"].map(
        {"1": True, "0": False, 1: True, 0: False, True: True, False: False}
    ).astype("boolean")
    for src_c, dst in [
        ("priority_string", "priority"), ("tags", "tags"), ("notes", "notes"),
        ("storage_string", "storage"),
        ("start_date_string", "start_date_raw"), ("finish_date_string", "finish_date_raw"),
    ]:
        d[dst] = clean_text_series(d[src_c])

    return pa.Table.from_pandas(
        d[[fl.name for fl in ESQUEMA_INTERACCIONES]],
        schema=ESQUEMA_INTERACCIONES,
        preserve_index=False,
    )


## 4.5 Construcción de la tabla de agregados por anime

Convierte los acumuladores en `anime_stats_globales.parquet`, una fila por anime con:

| Columna | Contenido |
|---|---|
| `n_interactions` | Filas totales del título |
| `n_scored`, `score_sum`, `mean_score` | Base de la nota media |
| `n_watching`, `n_completed`, `n_on_hold`, `n_dropped`, `n_plan_to_watch` | Reparto por estado |
| `score_1` … `score_10` | Histograma completo de puntuaciones |
| `sum_watched_episodes`, `n_rewatching` | Volumen de consumo |

**Solo se emiten los identificadores con al menos una interacción** (`np.nonzero`): los
acumuladores están dimensionados por el `anime_id` máximo, pero la numeración de MAL tiene
huecos y emitir todas las posiciones llenaría la tabla de ceros.

**La media se calcula, no se acumula.** Se guardan `score_sum` y `n_scored` por separado y
`mean_score` se deriva de ellos, con la división protegida para que un título sin
puntuaciones dé nulo y no una división por cero. Guardar los dos componentes permite
además recalcular medias sobre subconjuntos o combinar títulos.

**El histograma completo** (`score_1` … `score_10`) se conserva porque la media sola
esconde la forma de la distribución: una nota de 7 puede venir de un consenso o de la
mezcla de muchos 10 y muchos 4, y son fenómenos distintos.


In [ ]:
def _construir_stats_globales(acc, st):
    """Convierte los acumuladores en la tabla de agregados por anime."""
    maxid = acc["maxid"]
    # Solo los identificadores con actividad: la numeracion de MAL tiene huecos y
    # emitir todas las posiciones llenaria la tabla de ceros.
    ids = np.nonzero(acc["total"])[0]
    gs = acc["status"].reshape(maxid, NSTAT)
    gh = acc["hist"].reshape(maxid, NSCORE)

    gdf = pd.DataFrame({
        "anime_id": ids.astype("int32"),
        "n_interactions": acc["total"][ids],
        "n_scored": acc["scored"][ids],
        "score_sum": acc["scoresum"][ids],
        "sum_watched_episodes": acc["episodes"][ids],
        "n_rewatching": acc["rewatch"][ids],
    })
    # La media se deriva de suma y recuento, que se conservan: permite recalcularla
    # sobre subconjuntos y evita la division por cero de los titulos sin puntuar.
    gdf["mean_score"] = np.where(
        gdf["n_scored"] > 0,
        gdf["score_sum"] / gdf["n_scored"].where(gdf["n_scored"] > 0),
        np.nan,
    )
    for code, name in STATUS_LABELS.items():
        gdf[f"n_{name}"] = gs[ids, code]
    # El histograma completo se guarda porque la media esconde la forma de la
    # distribucion: un 7 de consenso y un 7 de mezcla de dieces y cuatros no son lo mismo.
    for s in range(SCORE_MIN, SCORE_MAX + 1):
        gdf[f"score_{s}"] = gh[ids, s]

    # Aviso de alcance: los agregados cubren TODAS las filas leidas del origen,
    # mientras que interacciones.parquet contiene solo la cohorte seleccionada.
    st["stats_scope"] = "todas las filas leidas del origen (no solo la cohorte escrita)"
    return gdf


## 4.6 Orquestador de la etapa 3

Junta todas las piezas en el bucle principal, que por cada row group:

1. lo lee con las columnas necesarias,
2. actualiza los acumuladores globales (**todas** las filas),
3. aplica el filtro de cohorte,
4. transforma y escribe las filas que quedan,
5. informa del progreso cada 25 bloques.

**Detalles de robustez**

- El `ParquetWriter` se cierra en un bloque `finally`. Si el proceso falla en el bloque
  600 de 924, el fichero escrito hasta ahí queda íntegro y legible en lugar de corrupto.
- `limit_rgs` permite ejecutar la etapa sobre unos pocos bloques para probar el flujo
  completo sin esperar horas.
- El progreso se informa con porcentaje y minutos transcurridos, porque es la única forma
  de saber si un proceso de varias horas avanza como debe.

**Coste real de la ejecución completa:** 968 millones de filas leídas, 527 millones
escritas, unos 8,4 GB de salida.


In [ ]:
def stage3_interactions(cohort_mode="favorites", limit_rgs=None):
    """Orquestador de la etapa 3."""
    log(f"ETAPA 3: interacciones (cohorte='{cohort_mode}')")
    st = REPORT["stages"]["3_interactions"] = {"cohort_mode": cohort_mode}

    f = pq.ParquetFile(F_ULIST)
    nrg = f.metadata.num_row_groups
    total_rows = f.metadata.num_rows
    rgs = list(range(nrg if limit_rgs is None else min(limit_rgs, nrg)))
    n_rgs = len(rgs)
    st["row_groups_scanned"] = n_rgs
    st["file_total_rows"] = total_rows

    cohort_idx = _cargar_cohorte(cohort_mode, st)

    # El tamano de los acumuladores se deriva del propio fichero: con un tope fijo
    # las filas con anime_id mayor se descartaban en silencio.
    maxid_seen = _max_anime_id(f)
    maxid = int(maxid_seen) + 1 if maxid_seen is not None else 200_000
    st["max_anime_id_in_source"] = maxid_seen
    log(f"  anime_id maximo en origen: {maxid_seen} -> acumuladores de tamano {maxid:,}")

    acc = _crear_acumuladores(maxid)
    counters = _nuevos_contadores()

    destino = os.path.join(OUT, "interacciones.parquet")
    writer = pq.ParquetWriter(destino, ESQUEMA_INTERACCIONES, compression=COMPRESSION)

    t0 = time.time()
    try:
        for i, rg in enumerate(rgs):
            tb = f.read_row_group(rg, columns=COLS_ULIST)
            counters["rows_seen"] += tb.num_rows

            ok_id = _acumular_bloque(acc, tb, counters)

            un = tb["username"].combine_chunks().to_pandas()
            if cohort_idx is not None:
                keep = un.isin(cohort_idx).to_numpy(dtype=bool)
                counters["username_not_in_cohort"] += int((~keep).sum())
            else:
                keep = np.ones(tb.num_rows, dtype=bool)
            keep = keep & ok_id

            if keep.any():
                out_tbl = _transformar_bloque(tb, keep, counters)
                writer.write_table(out_tbl)
                counters["rows_written"] += out_tbl.num_rows

            if i % 25 == 0 or i == n_rgs - 1:
                el = time.time() - t0
                pct = 100 * counters["rows_seen"] / total_rows
                log(
                    f"  rg {rg:>4}/{nrg}  {counters['rows_seen']:>13,} leidas ({pct:5.1f}%)  "
                    f"{counters['rows_written']:>13,} escritas  {el/60:5.1f} min"
                )
    finally:
        # Cerrar en `finally` deja el parquet integro aunque el proceso falle a mitad.
        writer.close()

    st["counters"] = counters
    REPORT.setdefault("outputs", {})["interacciones.parquet"] = {
        "rows": counters["rows_written"],
        "columns": len(ESQUEMA_INTERACCIONES),
        "size_mb": round(os.path.getsize(destino) / 1024**2, 2),
    }
    log(f"    -> interacciones.parquet {counters['rows_written']:,} filas, "
        f"{os.path.getsize(destino)/1024**3:.2f} GB")

    write(_construir_stats_globales(acc, st), "anime_stats_globales.parquet")
    log(f"  agregados globales sobre {counters['rows_seen']:,} filas")
    log("ETAPA 3 completada\n")


# ETAPA 4: INFORME FINAL Y COBERTURA

Cierra el pipeline volcando `REPORT` a `clean/_report.json` y añadiendo la métrica que no
puede calcularse hasta que todo lo demás existe: **cuánta de la actividad observada tiene
metadatos de catálogo**.


## 5.1 Informe y cálculo de cobertura

**La cobertura se mide de dos maneras distintas**, y la diferencia entre ambas es el
resultado interesante:

- **Por títulos**: cuántos `anime_id` de las interacciones aparecen en `anime.parquet`.
  Son 19.302 de 29.604, un 65%.
- **Por volumen de interacciones**: qué porcentaje de la actividad corresponde a esos
  títulos. Es el **98,4%**.

Los 10.302 títulos sin metadatos son material marginal —OVAs antiguas, especiales,
contenido retirado— que apenas nadie ve. Quedarse solo con el primer número llevaría a
pensar que falta un tercio del catálogo; el segundo muestra que el hueco es
prácticamente irrelevante para cualquier análisis ponderado por uso.

El cálculo va dentro de un `try`: si falta alguna tabla, el error se registra **dentro del
informe** en lugar de tumbar la ejecución. El informe siempre se escribe.


In [ ]:
def stage4_report():
    log("ETAPA 4: informe")
    p = os.path.join(OUT, "_report.json")
    try:
        anime = pq.read_table(os.path.join(OUT, "anime.parquet"), columns=["mal_id"]).to_pandas()
        g = pq.read_table(
            os.path.join(OUT, "anime_stats_globales.parquet"),
            columns=["anime_id", "n_interactions"],
        ).to_pandas()
        hit = g["anime_id"].isin(set(anime["mal_id"]))
        total = int(g["n_interactions"].sum())
        REPORT["coverage"] = {
            "anime_ids_in_interactions": int(len(g)),
            "with_anilist_metadata": int(hit.sum()),
            "without_anilist_metadata": int((~hit).sum()),
            "pct_interactions_covered": (
                round(100 * g.loc[hit, "n_interactions"].sum() / total, 4) if total else None
            ),
        }
        log(
            f"  cobertura de metadata: {hit.sum():,}/{len(g):,} anime, "
            f"{REPORT['coverage']['pct_interactions_covered']}% de las interacciones"
        )
    except Exception as e:
        REPORT["coverage"] = {"error": f"{type(e).__name__}: {e}"}
        log(f"  no se pudo calcular la cobertura: {e}")
    with open(p, "w", encoding="utf-8") as fh:
        json.dump(REPORT, fh, indent=2, ensure_ascii=False, default=str)
    log(f"  -> {p}")


## 5.2 Ejecución del pipeline

`main()` permite ejecutar etapas sueltas (`stages="1,2"`), cambiar la cohorte o limitar el
número de row groups para una prueba rápida.

**La llamada está comentada a propósito.** La etapa 3 recorre 968 millones de filas y
tarda horas; ninguna celda de este notebook debe lanzar ese proceso por el hecho de
ejecutar todas las celdas de arriba abajo.

Para una prueba de humo del flujo completo sin esperar:

```python
main(stages="3", cohort="favorites", limit_row_groups=5)
```

**Dependencias entre etapas:** la 3 necesita la cohorte que produce la 2, y la 4 necesita
las salidas de la 1 y la 3. La 1 y la 2 son independientes entre sí.


In [ ]:
def main(stages="1,2,3,4", cohort="favorites", limit_row_groups=None):
    stages = {s.strip() for s in str(stages).split(",")}
    if "1" in stages:
        stage1_anime()
    if "2" in stages:
        stage2_users()
    if "3" in stages:
        stage3_interactions(cohort, limit_row_groups)
    if "4" in stages:
        stage4_report()
    log("LISTO")

# Ejecuta todas las etapas (descomentar; la etapa 3 recorre ~968M de filas y tarda horas)
# main(stages="1,2,3,4", cohort="favorites", limit_row_groups=None)


# AUDITORÍA DE NULOS Y POLÍTICA DE REPARACIÓN

A partir de aquí ya no se genera nada: se audita lo generado.

**La regla de negocio que se aplica al revisar los nulos**

- Se mantiene el nulo cuando el valor es realmente desconocido o no se puede inferir.
- No se inventan valores si no hay evidencia en el origen.
- Las columnas opcionales se conservan como campo «no disponible», no se eliminan.
- Solo se repara un valor cuando existe una regla estricta y verificable que lo justifique.

**Cómo leer los porcentajes de nulos.** Un 95% de nulos en `birth_year` no es un defecto
del pipeline: es que la inmensa mayoría de usuarios no publica su fecha de nacimiento. Un
99% de nulos en `storage` significa que casi nadie rellena ese campo opcional. El objetivo
de la auditoría es **distinguir el nulo estructural del nulo por error**, no minimizar el
recuento.


## 6.1 Utilidades de auditoría

**`null_summary(df)`** — devuelve las columnas con nulos, ordenadas por volumen, con su
porcentaje y su tipo. Omite las columnas sin nulos para que el resultado quepa en
pantalla, y devuelve un DataFrame vacío bien formado cuando no hay ninguna, en lugar de
fallar.

**`report_table(name, path, sample_row_groups)`** — el punto importante de esta celda.

Las tablas se auditan de dos formas según su tamaño:

| Modo | Tablas | Cómo |
|---|---|---|
| Completo | Catálogo y derivadas (decenas de miles de filas) | Se cargan enteras |
| Muestreado | `users`, `user_texto`, `interacciones` | Un row group representativo |

`users.parquet` ocupa 1 GB, `user_texto.parquet` 1,2 GB e `interacciones.parquet` 8,4 GB:
cargarlas enteras para contar nulos es innecesario y en el último caso imposible. **El
número total de filas se toma de los metadatos** —siempre exacto— y solo el detalle de
nulos se calcula sobre la muestra. La salida indica siempre qué modo se ha usado, para
que nadie lea un porcentaje muestral como si fuera censal.


In [ ]:
# Auditoria de nulos sobre los parquet ya generados.
# Se leen solo metadatos y columnas necesarias: users.parquet (1 GB) y
# user_texto.parquet no caben comodos en memoria si se cargan enteros.
import os
import pandas as pd
import pyarrow.parquet as pq

AUDIT_BASE = r"D:\Codigos\Visualizacion\dataset 2026\clean"

FULL_LOAD = {
    "anime": "anime.parquet",
    "anime_stats": "anime_stats_globales.parquet",
    "personajes": "anime_personajes_anilist.parquet",
    "rankings": "anime_rankings.parquet",
    "recomendaciones": "anime_recomendaciones.parquet",
    "relaciones": "anime_relaciones.parquet",
    "generos": "anime_generos.parquet",
}
# Tablas grandes: se audita una muestra por row group en lugar del fichero completo.
SAMPLED = {
    "users": "users.parquet",
    "texto": "user_texto.parquet",
    "interacciones": "interacciones.parquet",
}


def null_summary(df):
    n = len(df)
    if n == 0:
        return pd.DataFrame(columns=["column", "nulls", "pct", "dtype"])
    out = []
    for col in df.columns:
        cnt = int(df[col].isna().sum())
        if cnt == 0:
            continue
        out.append({"column": col, "nulls": cnt, "pct": round(100 * cnt / n, 2), "dtype": str(df[col].dtype)})
    if not out:
        return pd.DataFrame(columns=["column", "nulls", "pct", "dtype"])
    return pd.DataFrame(out).sort_values(["nulls", "column"], ascending=[False, True]).reset_index(drop=True)


def report_table(name, path, sample_row_groups=None):
    if not os.path.exists(path):
        print(f"=== {name} === FALTA: {path}\n")
        return
    pf = pq.ParquetFile(path)
    total = pf.metadata.num_rows
    if sample_row_groups is None:
        df = pd.read_parquet(path)
        origen = "tabla completa"
    else:
        n = min(sample_row_groups, pf.metadata.num_row_groups)
        df = pd.concat([pf.read_row_group(i).to_pandas() for i in range(n)], ignore_index=True)
        origen = f"muestra de {n}/{pf.metadata.num_row_groups} row groups"
    print(f"=== {name} ===")
    print(f"filas totales={total:,}  columnas={pf.metadata.num_columns}  ({origen}: {len(df):,} filas)")
    s = null_summary(df)
    print("sin nulos" if s.empty else s.head(12).to_string(index=False))
    print()


## 6.2 Ejecución de la auditoría

Recorre las tablas de `clean/` e imprime, para cada una, sus dimensiones reales y su
reparto de nulos.

**Qué buscar en la salida**

- **Nulos estructurales esperados**: `birth_*` (~95%), `location` (~94%), `storage`,
  `notes`, `tags` (>96%). Son campos opcionales del perfil o de la lista. Correcto.
- **`score` con ~34% de nulos** en interacciones: usuarios que tienen el anime en su lista
  pero no lo han puntuado. Es el dato real y hay que tenerlo presente al calcular medias.
- **`start_date` con ~89%**: la mayoría no registra fechas de visionado. La comparación con
  `start_date_raw` (~86%) indica cuántas venían con texto pero no se pudieron interpretar.
- **Columnas que deberían tener pocos nulos y tienen muchos**: eso sí sería un síntoma de
  fallo en el pipeline.

Si esta celda se ejecuta antes de volver a lanzar las etapas 2 y 3, `user_texto` seguirá
mostrando los 18,2 millones de filas de la versión anterior, no el resultado corregido.


In [ ]:
for name, fname in FULL_LOAD.items():
    report_table(name, os.path.join(AUDIT_BASE, fname))
for name, fname in SAMPLED.items():
    report_table(name, os.path.join(AUDIT_BASE, fname), sample_row_groups=1)


# BLOQUE ANALÍTICO: PREPARACIÓN Y ESTUDIO

A partir de aquí se trabaja sobre los parquet ya generados en `clean/`. Ninguna celda de
este bloque vuelve a leer las fuentes originales.

**La restricción que condiciona todo el bloque**

`interacciones.parquet` tiene **527 millones de filas y 8,4 GB**. No cabe en memoria.
Cualquier análisis que dependa de esa tabla se agrega recorriéndola por row group con
`iter_interacciones()`, y solo el resultado agregado —de tamaño manejable— se guarda en
`clean/analysis/`.

Por eso **no existe** una tabla usuario-anime materializada: a ese nivel de detalle el
análisis se hace en streaming, no cargando.

**Las dos escalas de puntuación.** Conviven en el dataset y no son intercambiables:

| Columna | Origen | Escala |
|---|---|---|
| `average_score`, `meanScore` | AniList | **0-100** |
| `mean_score` | Listas de usuarios de MAL (etapa 3) | **1-10** |
| `anime_mean` | Media declarada en el perfil | **1-10** |

Cada gráfico indica en su eje cuál está usando.


## 7.1 Configuración del bloque analítico

Rutas, estilo y **umbrales explícitos**. Se define una sola vez y todas las celdas
siguientes lo reutilizan, en lugar de repetir la ruta base en cada una.

**Los umbrales de audiencia son la parte importante**

| Constante | Valor | Para qué |
|---|---|---|
| `MIN_POPULARITY` | 5.000 | Usuarios de AniList con el título en su lista |
| `MIN_SCORED` | 1.000 | Usuarios de MAL que puntuaron el título |

Un ranking «de los mejores» sin mínimo de audiencia no mide calidad, mide **ruido
estadístico**: los primeros puestos los ocupan títulos con dos o tres valoraciones
perfectas. Con el umbral aplicado, el top 10 lo forman los títulos que uno esperaría
—Frieren, Fullmetal Alchemist: Brotherhood, Steins;Gate, Hunter x Hunter—, que es la señal
de que el filtro hace lo que debe.

Los umbrales son constantes con nombre, no números escondidos dentro de una función, para
que cambiarlos sea una decisión visible y para que los gráficos puedan citarlos en su
título.

**`GENDER_LABELS` y su advertencia.** La fuente no documenta el significado de los códigos
de género. El mapeo que se usa es la interpretación habitual del volcado, y por eso va
acompañado de `GENDER_NOTE`, que se imprime **dentro del propio gráfico**. Conviene
validarlo antes de sacar conclusiones.

**`SNAPSHOT_YEAR`** se lee de `_report.json`. Las edades se calculan contra el año del
snapshot, no contra un año escrito a mano en el código, que quedaría obsoleto en cuanto
pasara enero.


In [ ]:
# Configuracion compartida para todo el bloque de visualizacion y analisis.
# Las celdas siguientes reutilizan estas constantes y helpers en vez de redefinir rutas.
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

VIZ_BASE = Path(r"D:\Codigos\Visualizacion\dataset 2026")
CLEAN = VIZ_BASE / "clean"
ANALYSIS = CLEAN / "analysis"
CHART_DIR = CLEAN / "charts"
ANALYSIS.mkdir(exist_ok=True, parents=True)
CHART_DIR.mkdir(exist_ok=True, parents=True)

# Los datasets de los demas anos viven un nivel por encima de esta carpeta.
RAIZ = VIZ_BASE.parent
PANEL = RAIZ / "panel_temporal"                      # 9 cortes normalizados, 2018-2026
USUARIOS_2023 = RAIZ / "dataset 2024/users-details-2023.csv"
VALORAC_2023 = RAIZ / "dataset 2024/users-score-2023.csv"

# Rampa ordinal de un solo tono para series ordenadas en el tiempo (cortes del panel).
# Validada: L monotona, saltos >= 0.06 y el extremo claro por encima de 2:1.
RAMPA_CORTES = ["#86b6ef", "#5598e7", "#2a78d6", "#184f95"]

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 220

# Umbrales explicitos: un ranking "de los mejores" sin minimo de votos devuelve
# titulos anecdoticos con dos o tres valoraciones perfectas.
MIN_POPULARITY = 5_000     # usuarios de AniList con el titulo en su lista
MIN_SCORED = 1_000         # usuarios de MAL que puntuaron el titulo en este dataset

# El codigo de `gender` no viene documentado por la fuente, pero se ha podido
# DEDUCIR. El volcado de 2023 (dataset 2024/users-details-2023.csv) trae el genero
# como texto y comparte usuarios con este; cruzando 222.619 nombres:
#     codigo 1 -> Male        100,0%  (125.487 casos)
#     codigo 2 -> Female       99,9%  ( 95.576 casos)
#     codigo 3 -> Non-Binary   89,4%  (    933 casos)
#     codigo 0 -> sin senal, 623 casos repartidos entre ambos: es "sin especificar"
# El mapeo anterior tenia 1 y 2 intercambiados, asi que el grafico 8.4 etiquetaba
# al reves las barras de masculino y femenino.
GENDER_LABELS = {0: "Sin especificar", 1: "Masculino", 2: "Femenino", 3: "No binario"}
GENDER_NOTE = ("Códigos de género deducidos cruzando 222.619 usuarios con el volcado de 2023, "
               "que los trae como texto.")

# Ano de referencia del snapshot, para no dejar el ano escrito a mano en el codigo.
def _snapshot_year(default=None):
    p = CLEAN / "_report.json"
    if p.exists():
        try:
            s = json.loads(p.read_text(encoding="utf-8"))["stages"]["2_users"]["snapshot_utc"]
            return int(pd.Timestamp(s).year)
        except Exception:
            pass
    return default or int(pd.Timestamp.today().year)

SNAPSHOT_YEAR = _snapshot_year()


## 7.2 Utilidades de presentación

**`savefig(name, fig)`** — guarda el gráfico en `clean/charts/` y lo muestra en el
notebook. La versión anterior forzaba el backend `Agg`, que escribe ficheros pero **no
dibuja nada en pantalla**: había que abrir el PNG a mano para ver el resultado.

**`fmt_miles(x)`** — formatea el eje eligiendo la escala según la magnitud real del dato
(k, M, B). Sustituye a un formateador que dividía siempre entre un millón, lo que
convertía cualquier eje con valores menores en una fila de ceros.

**`nota_media_valida(df)`** — implementa una corrección con consecuencias analíticas
reales. `anime_mean` vale `0` cuando el perfil **no ha puntuado ningún anime**: MyAnimeList
muestra `0.00` en ese caso. Son 10,4 millones de perfiles y 13,9% de los usuarios activos.

Tomar ese cero como una nota real:

| | Media de los usuarios activos |
|---|---|
| Con los ceros | 7,09 |
| Sin los ceros | **8,23** |

La función descarta los ceros y las medias marcadas como fuera de escala en la etapa 2, y
devuelve el resto. Todas las celdas que usan la nota del perfil pasan por aquí.


In [ ]:
def savefig(name, fig, show=True):
    path = CHART_DIR / name
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    print(f"guardado: {path}")
    if show:
        plt.show()
    plt.close(fig)


def fmt_miles(x, _pos=None):
    """Formateador de eje que elige la escala segun la magnitud real de los datos."""
    ax = abs(x)
    if ax >= 1e9:
        return f"{x/1e9:.1f}B"
    if ax >= 1e6:
        return f"{x/1e6:.1f}M"
    if ax >= 1e3:
        return f"{x/1e3:.0f}k"
    return f"{x:.0f}"


def nota_media_valida(df, col="anime_mean"):
    """Nota media declarada por el usuario, con los valores no interpretables como nulos.

    `anime_mean` vale 0 cuando el perfil no tiene ninguna puntuacion: MAL muestra
    0.00 en ese caso. Tomarlo como una nota real hunde cualquier media por segmento
    (8.23 frente a 7.09 sobre los usuarios activos de este dataset).
    """
    s = pd.to_numeric(df[col], errors="coerce")
    flag = f"{col}_invalid"
    fuera = df[flag].fillna(False) if flag in df.columns else False
    return s.where((s > 0) & ~fuera)


## 7.3 Lectores de datos

**`load_anime_catalogo()`** — carga el catálogo con las columnas de análisis y resuelve
dos problemas de la tabla original:

1. **`anime.parquet` no tiene columna `title`.** Tiene cuatro variantes de título
   (`title_english`, `title_romaji`, `title_native`, `title_userPreferred`) y cualquiera
   puede faltar. Se construye un `title` único con una cascada de preferencia
   inglés → romaji → nativo y, si faltan los tres, se usa `mal_id N` como último recurso,
   para que ningún título quede sin etiqueta en un gráfico.
2. **Renombra `averageScore` a `average_score`** y expone `year`, dando nombres estables
   al resto del bloque.

**`load_stats_globales()`** — los agregados por anime de la etapa 3, en escala 1-10.

**`iter_interacciones(columns, max_row_groups)`** — el generador que sustituye a
`pd.read_parquet` sobre el fichero grande. Entrega un DataFrame por row group leyendo
**solo las columnas pedidas**, que es lo que hace viable recorrer 527 millones de filas.
`max_row_groups` permite trabajar sobre una muestra durante la exploración.

**`nota_muestreo(max_row_groups)`** — genera el texto que advierte del muestreo. Si un
gráfico se calcula sobre 20 de 924 bloques, **eso tiene que verse en el gráfico**, no
quedarse en el argumento de una función. Devuelve cadena vacía cuando se ha recorrido todo.


In [ ]:
def load_anime_catalogo():
    """Catalogo de anime con un titulo unico resuelto por preferencia de idioma."""
    a = pd.read_parquet(
        CLEAN / "anime.parquet",
        columns=["mal_id", "title_english", "title_romaji", "title_native",
                 "type", "format", "episodes", "averageScore", "meanScore",
                 "popularity", "favourites", "startDate_year"],
    )
    a["title"] = a["title_english"].fillna(a["title_romaji"]).fillna(a["title_native"])
    a["title"] = a["title"].fillna("mal_id " + a["mal_id"].astype(str))
    a["average_score"] = pd.to_numeric(a["averageScore"], errors="coerce")   # escala AniList 0-100
    a["popularity"] = pd.to_numeric(a["popularity"], errors="coerce")
    a["year"] = pd.to_numeric(a["startDate_year"], errors="coerce")
    return a


def load_stats_globales():
    """Agregados por anime calculados en la etapa 3 (escala de nota 1-10 de MAL)."""
    return pd.read_parquet(CLEAN / "anime_stats_globales.parquet")


def iter_interacciones(columns, max_row_groups=None):
    """Recorre interacciones.parquet por row group.

    El fichero tiene ~527M de filas y 8.4 GB: no se puede cargar entero con
    pd.read_parquet. Cualquier analisis sobre interacciones debe agregarse por bloques.
    """
    pf = pq.ParquetFile(CLEAN / "interacciones.parquet")
    n = pf.metadata.num_row_groups if max_row_groups is None else min(max_row_groups, pf.metadata.num_row_groups)
    for rg in range(n):
        yield pf.read_row_group(rg, columns=columns).to_pandas()


def load_panel(anos=None):
    """Cortes anuales normalizados de `panel_temporal/`, indexados por `anime_id`.

    Un corte por ano entre 2018 y 2026, todos con las mismas columnas. Sirve para
    comparar el MISMO titulo medido en momentos distintos, que es lo unico que
    permite separar "esto ha cambiado" de "esto siempre fue asi".

    Aviso de comparabilidad: 2020 y 2026 traen una media calculada por nosotros y
    no la nota publicada por MyAnimeList; quedan 0,11 y 0,06 puntos por debajo de
    lo que cabria esperar. Para comparar entre cortes, usar rangos o asumir ese
    desfase.
    """
    if not PANEL.exists():
        raise FileNotFoundError(f"Falta {PANEL}. Ejecuta antes construir_panel.py")
    disponibles = sorted(int(f.stem.split("_")[1]) for f in PANEL.glob("anime_*.parquet"))
    anos = disponibles if anos is None else [a for a in anos if a in disponibles]
    salida = {}
    for a in anos:
        d = pd.read_parquet(PANEL / f"anime_{a}.parquet").dropna(subset=["anime_id"])
        d["anime_id"] = d["anime_id"].astype(int)
        salida[a] = d.drop_duplicates("anime_id").set_index("anime_id")
    return salida


def load_usuarios_2023(edad_min=10, edad_max=80, ano_ref=2023):
    """Perfiles del volcado de 2023, con genero en TEXTO y edad ya depurada.

    Es la unica fuente del proyecto donde el genero viene escrito y no como codigo,
    y ademas cabe entera en memoria (731.290 perfiles), asi que evita el muestreo
    que necesitan las tablas de 2026.
    """
    u = pd.read_csv(USUARIOS_2023, usecols=["Mal ID", "Username", "Gender", "Birthday"],
                    low_memory=False).rename(columns={"Mal ID": "user_id"})
    nac = pd.to_datetime(u["Birthday"], errors="coerce", format="mixed", utc=True)
    u["edad"] = ano_ref - nac.dt.year
    # El campo admite cualquier cosa: hay anos entre 1 y 2100. Fuera del rango
    # razonable son bromas o errores de captura, no edades.
    u.loc[~u["edad"].between(edad_min, edad_max), "edad"] = np.nan
    u["franja"] = (u["edad"] // 10 * 10)
    return u[["user_id", "Username", "Gender", "edad", "franja"]]


def load_valoraciones_2023(columnas=("user_id", "anime_id"), chunk=5_000_000):
    """Recorre por bloques las ~30,4 M de valoraciones del volcado de 2023."""
    for ch in pd.read_csv(VALORAC_2023, usecols=list(columnas), chunksize=chunk):
        yield ch


def ids_primera_temporada():
    """`mal_id` de los titulos que no son continuacion ni derivado de otro.

    Se deduce de `anime_relaciones.parquet`, no del titulo: un titulo con una
    relacion PREQUEL es una secuela, y uno con PARENT es un derivado (especial,
    recopilatorio, spin-off). Es mas fiable que buscar "Season 2" en el nombre,
    que falla con "Gintama'", "Shippuuden" o los numerales romanos.

    Comprobado: Kaguya-sama: Love is War entra y su segunda temporada no;
    Hunter x Hunter (2011) entra, por ser reinicio y no continuacion.
    """
    r = pd.read_parquet(CLEAN / "anime_relaciones.parquet",
                        columns=["mal_id_from", "relation_type"])
    derivados = set(r.loc[r["relation_type"].isin(["PREQUEL", "PARENT"]),
                          "mal_id_from"].dropna().astype(int))
    todos = set(pd.read_parquet(CLEAN / "anime.parquet", columns=["mal_id"])["mal_id"])
    return todos - derivados


def nota_muestreo(max_row_groups):
    if max_row_groups is None:
        return ""
    pf = pq.ParquetFile(CLEAN / "interacciones.parquet")
    total = pf.metadata.num_row_groups
    if max_row_groups >= total:
        return ""
    return f"\n(muestra parcial: {max_row_groups} de {total} bloques de interacciones)"


print(f"CLEAN   = {CLEAN}")
print(f"CHARTS  = {CHART_DIR}")
print(f"Ano de referencia del snapshot: {SNAPSHOT_YEAR}")


# GRÁFICOS EXPLORATORIOS

Seis vistas sobre el catálogo y sobre el comportamiento de los usuarios. Cada una se
guarda en `clean/charts/` y se muestra a continuación de su celda.

**Criterio común a todos los gráficos de este bloque**

- Todo ranking de calidad lleva un **mínimo de audiencia** declarado en el título.
- Todo cálculo sobre una muestra lo **dice en el título**, no en un comentario del código.
- Los ejes indican siempre la escala de puntuación que se está usando.


## 8.1 Géneros más frecuentes del catálogo

Recuento de títulos por género sobre `anime_generos.parquet`.

**Qué mide exactamente:** cuántos **títulos** llevan cada etiqueta, no cuánto se ven. Un
género puede encabezar esta lista por tener muchas producciones pequeñas. La popularidad
medida por consumo es otra cosa y se ve en los gráficos siguientes.

Como un anime tiene varios géneros, la suma de las barras supera el número de títulos del
catálogo: no son categorías excluyentes.


In [ ]:
def chart_categorias_populares(top_n=15):
    generos = pd.read_parquet(CLEAN / "anime_generos.parquet")
    top = generos["genre"].value_counts().head(top_n).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(top.index, top.values, color="#4C78A8")
    ax.set_title(f"Top {top_n} géneros por número de títulos en el catálogo")
    ax.set_xlabel("Número de animes")
    ax.set_ylabel("Género")
    ax.set_xlim(0, top.max() * 1.10)
    for y, v in enumerate(top.values):
        ax.text(v, y, f" {v:,}", va="center", fontsize=9)
    savefig("categorias_mas_populares.png", fig)

chart_categorias_populares()


## 8.2 Top 10 mejor valorados, en las dos plataformas

**El umbral sigue siendo la mitad del gráfico.** Sin `MIN_POPULARITY`, el ranking lo copan
títulos con dos o tres valoraciones perfectas, que no son comparables con obras valoradas
por cientos de miles de personas. El filtro deja 5.377 títulos elegibles de casi 20.000.

**La otra mitad es la validación cruzada.** La versión anterior mostraba solo el ranking de
AniList, y no había forma de saber si ese top era una peculiaridad de esa plataforma. Ahora
los dos rankings van uno al lado del otro y **se colorean los títulos que aparecen en
ambos**: coinciden 8 de 10, así que el ranking es sólido y no un artefacto de la fuente.

Las dos escalas son distintas —AniList 0-100, MyAnimeList 1-10— y las capturas están
separadas (agosto de 2026 y mayo de 2024), así que **no se comparan valores: se comparan
posiciones**. Por eso el gráfico no intenta ponerlas en un eje común.


In [ ]:
def chart_top_10_animes_mejor_valorados(top_n=10, min_popularity=MIN_POPULARITY,
                                        min_scored=10_000):
    """Top 10 según AniList y según MyAnimeList, uno al lado del otro.

    La versión anterior mostraba solo el ranking de AniList. El problema no era el
    umbral —ese ya estaba— sino que no había forma de saber si ese top era una
    peculiaridad de AniList. Poniendo los dos rankings juntos se ve de inmediato
    cuánto coinciden, y el lector no tiene que fiarse de una sola fuente.

    Las dos escalas son distintas (AniList 0-100, MyAnimeList 1-10) y las capturas
    están separadas, así que **no se comparan valores: se comparan posiciones**.
    """
    anime = load_anime_catalogo()
    elegibles = anime[anime["average_score"].notna() & (anime["popularity"] >= min_popularity)]
    top_ani = elegibles.nlargest(top_n, "average_score")

    mal = load_panel([2024])[2024]
    top_mal = mal[mal["scored_by"] >= min_scored].nlargest(top_n, "score")

    comunes = set(top_ani["mal_id"]) & set(top_mal.index)
    print(f"{len(elegibles):,} títulos superan el mínimo de {min_popularity:,} usuarios en AniList")
    print(f"coinciden {len(comunes)} de {top_n} entre los dos rankings")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    for ax, datos, col, lim, titulo, sub in [
        (ax1, top_ani.assign(v=top_ani["average_score"], t=top_ani["title"]), "#F58518", 100,
         "AniList", "Puntuación media (0-100)"),
        (ax2, top_mal.assign(v=top_mal["score"], t=top_mal["title"]), "#4C78A8", 10,
         "MyAnimeList", "Nota media de las listas (1-10)"),
    ]:
        d = datos.iloc[::-1]
        ids = d["mal_id"] if "mal_id" in d.columns else d.index
        colores = [col if i in comunes else "#c3c2b7" for i in ids]
        ax.barh(range(len(d)), d["v"], color=colores, height=0.66)
        ax.set_yticks(range(len(d)))
        ax.set_yticklabels([str(t)[:34] for t in d["t"]], fontsize=9)
        for i, v in enumerate(d["v"]):
            ax.text(v + lim * 0.012, i, f"{v:.0f}" if lim == 100 else f"{v:.2f}",
                    va="center", fontsize=9, fontweight="bold")
        ax.set_xlim(0, lim * 1.18)
        ax.set_title(titulo, fontsize=12, fontweight="bold", loc="left", pad=10)
        ax.set_xlabel(sub, fontsize=9)
        ax.xaxis.grid(True)
        ax.set_axisbelow(True)
        for e in ("top", "right", "left"):
            ax.spines[e].set_visible(False)

    fig.suptitle(f"Los dos rankings coinciden en {len(comunes)} de {top_n} títulos\n"
                 f"En color, los que aparecen en ambos; en gris, los exclusivos de cada plataforma",
                 fontsize=14, fontweight="bold", x=0.045, ha="left", y=1.02)
    fig.text(0.045, -0.02,
             f"AniList: mínimo {min_popularity:,} usuarios con el título en su lista, captura de "
             f"agosto de 2026. MyAnimeList: mínimo {min_scored:,} votos, captura de mayo de 2024.\n"
             "Las escalas no son comparables entre sí; lo que se compara son las posiciones. "
             "Los títulos difieren\nporque cada plataforma usa su propia nomenclatura (Frieren / Sousou no "
             "Frieren, Gintama Season 3 / Gintama°): el emparejamiento se hace por identificador.",
             fontsize=8, color="#898781", va="top")
    savefig("top10_animes_mejor_valorados.png", fig)

chart_top_10_animes_mejor_valorados()


## 8.3 Preferencia de género por franja de edad

**Este gráfico se ha rehecho dos veces, y las dos por el mismo motivo: decía menos de lo que
parecía.**

**Cambio de fuente.** La versión anterior cruzaba perfiles de 2026 con 20 de los 924 bloques
de `interacciones.parquet` — **un 2% de las filas**. El volcado de 2023 (`dataset 2024`) trae
731.290 perfiles y 30,4 M de valoraciones que **caben enteros**, así que ahora se recorre el
**censo completo**: 166.551 perfiles con edad utilizable y 33,4 M de valoraciones cruzadas.
Se acabó el muestreo.

**Cambio de forma.** Con los datos completos resultó que **Action es el género más visto en
todas las franjas de edad**, así que un gráfico de «género dominante» dibujaba cinco barras
iguales que solo medían el tamaño de cada grupo. Lo que responde a la pregunta es cuánto se
**desvía** cada franja del reparto general: azul si ve ese género más que la media, rojo si
lo ve menos.

**Lo que aparece al mirarlo así:** la ciencia ficción es lo más generacional del catálogo
—+2,1 puntos en los cuarentones y +2,0 en los cincuentones, frente a −1,4 en los
veinteañeros—, mientras que comedia y romance pesan más entre los más jóvenes.

**Sobre la edad.** El campo de cumpleaños admite cualquier cosa: hay años entre 1 y 2100. Se
limita al rango 10-80 y se descartan las franjas con menos de 500 perfiles, que a partir de
los 60 son casi todas fechas de broma.


In [ ]:
def _cruce_demografico(campo, etiquetas=None, max_bloques=None, min_perfiles=500):
    """Recuento de (grupo demográfico × género de anime) sobre el volcado de 2023.

    Recorre las ~30,4 M de valoraciones por bloques, cruzando cada una con el perfil
    del usuario y con los géneros del título. Solo se acumula el recuento, así que
    en memoria nunca hay más de un bloque.

    Frente a la versión anterior, que leía 20 de los 924 bloques de
    `interacciones.parquet` —un 2% de las filas—, esta recorre el **100%** de una
    fuente que cabe entera, y usa el género de usuario **en texto** en vez de un
    código deducido.
    """
    u = load_usuarios_2023()
    u = u.dropna(subset=[campo])
    if etiquetas is not None:
        u = u[u[campo].isin(etiquetas)]
    # Los grupos diminutos son ruido: la franja de 70 anos tiene 140 perfiles y la
    # de 80 solo 12, casi todos fechas de nacimiento de broma.
    grandes = u[campo].value_counts()
    descartados = grandes[grandes < min_perfiles]
    if len(descartados):
        print(f"  grupos descartados por tener menos de {min_perfiles} perfiles: "
              f"{descartados.to_dict()}")
        u = u[u[campo].isin(grandes[grandes >= min_perfiles].index)]
    perfiles = u.set_index("user_id")[campo]

    gen = pd.read_parquet(PANEL / "anime_2023.parquet", columns=["anime_id", "genres"]).dropna()
    gen["anime_id"] = gen["anime_id"].astype(int)
    gen = (gen.assign(genero=gen["genres"].astype("string").str.split(", "))
              .explode("genero")[["anime_id", "genero"]])

    acumulado, filas, bloques = None, 0, 0
    for ch in load_valoraciones_2023():
        bloques += 1
        ch = ch[ch["user_id"].isin(perfiles.index)]
        if ch.empty:
            continue
        ch["grupo"] = ch["user_id"].map(perfiles)
        m = ch.merge(gen, on="anime_id", how="inner")
        filas += len(m)
        c = m.groupby(["grupo", "genero"], observed=True).size()
        acumulado = c if acumulado is None else acumulado.add(c, fill_value=0)
        if max_bloques is not None and bloques >= max_bloques:
            break

    if acumulado is None:
        raise ValueError("No se pudo construir el cruce demográfico.")
    conteo = acumulado.reset_index(name="count")
    total = conteo.groupby("grupo")["count"].sum()
    dominante = (conteo.sort_values(["grupo", "count"], ascending=[True, False])
                       .drop_duplicates("grupo"))
    dominante["share"] = dominante["count"].values / total.loc[dominante["grupo"]].values
    print(f"{len(perfiles):,} perfiles · {filas:,} valoraciones cruzadas · "
          f"{bloques} bloques leídos" + (" (PARCIAL)" if max_bloques else " (censo completo)"))
    return conteo, dominante.sort_values("count", ascending=False)


def _grafico_desviacion(campo, orden=None, etiquetas=None, titulo="", pie="",
                       nombre_fichero="", n_generos=6, min_perfiles=500):
    """Small multiples: cuánto se desvía cada grupo del reparto general de géneros.

    Un grupo que ve un 16% de acción y otro un 14% no se distinguen mirando cuál es
    su género dominante —es acción en los dos—, pero sí mirando la distancia al
    reparto global. Eso es lo que dibuja esta función.

    Codificación divergente: azul cuando el grupo consume ese género **más** que la
    media, rojo cuando lo consume **menos**, y el cero como referencia.
    """
    conteo, _ = _cruce_demografico(campo, min_perfiles=min_perfiles)
    piv = conteo.pivot(index="genero", columns="grupo", values="count").fillna(0)
    cuota = 100 * piv / piv.sum()
    global_ = 100 * piv.sum(axis=1) / piv.sum().sum()
    desv = cuota.sub(global_, axis=0)

    grupos = orden if orden is not None else list(desv.columns)
    grupos = [g for g in grupos if g in desv.columns]
    lim = max(abs(desv[grupos].values.min()), abs(desv[grupos].values.max())) * 1.35

    fig, axes = plt.subplots(1, len(grupos), figsize=(3.6 * len(grupos), 6.4), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, g in zip(axes, grupos):
        s = desv[g].reindex(global_.nlargest(18).index).dropna()
        s = pd.concat([s.nlargest(n_generos), s.nsmallest(n_generos)])
        s = s[~s.index.duplicated()].sort_values()
        ax.barh(range(len(s)), s.values,
                color=["#2a78d6" if v > 0 else "#d03b3b" for v in s.values], height=0.66)
        ax.set_yticks(range(len(s)))
        ax.set_yticklabels(s.index, fontsize=8.5)
        for i, v in enumerate(s.values):
            ax.text(v + (0.06 if v > 0 else -0.06), i, f"{v:+.1f}", va="center",
                    ha="left" if v > 0 else "right", fontsize=8, color="#0b0b0b")
        ax.axvline(0, color="#c3c2b7", linewidth=1)
        ax.set_xlim(-lim, lim)
        ax.xaxis.grid(True)
        ax.set_axisbelow(True)
        for e in ("top", "right", "left"):
            ax.spines[e].set_visible(False)
        et = etiquetas.get(g, g) if etiquetas else g
        ax.set_title(et, fontsize=11, fontweight="bold", loc="left", pad=8)
    for ax in axes:
        ax.set_xlabel("Desviación (puntos %)", fontsize=8.5)

    fig.suptitle(titulo, fontsize=14, fontweight="bold", x=0.06, ha="left", y=1.04)
    fig.text(0.06, -0.04, pie, fontsize=8, color="#898781", va="top")
    savefig(nombre_fichero, fig)
    return desv


def chart_categoria_por_edad():
    """Preferencia de género por franja de edad, sobre el censo de 2023."""
    desv = _grafico_desviacion(
        "franja",
        orden=[10.0, 20.0, 30.0, 40.0, 50.0],
        etiquetas={10.0: "10-19 años", 20.0: "20-29", 30.0: "30-39", 40.0: "40-49", 50.0: "50-59"},
        titulo="La ciencia ficción es lo más generacional del catálogo\n"
               "Desviación de cada franja de edad frente al reparto general de géneros",
        pie="Censo completo del volcado de 2023: 166.551 perfiles con edad utilizable y 33,4 M de "
            "valoraciones.\nAzul, el grupo ve ese género más que la media; rojo, menos. Se descartan "
            "las franjas con menos de 500 perfiles: las de 60 años en adelante son casi todas fechas "
            "de nacimiento de broma.",
        nombre_fichero="categoria_por_edad.png")
    print(desv.round(2).to_string())

chart_categoria_por_edad()


## 8.4 Preferencia de género según el género declarado del usuario

Misma mecánica que el gráfico anterior, y las mismas dos mejoras: **censo completo** en vez
de muestra (224.383 perfiles y 48,6 M de valoraciones) y **desviación** en vez de dominante.

**Aquí desaparece la advertencia que arrastraba este gráfico.** En el volcado de 2026 el
género del usuario es un código entero que la fuente no documenta; en el de 2023 viene
escrito (`Male`, `Female`, `Non-Binary`), así que no hay nada que interpretar. Es además la
fuente con la que se dedujo el mapeo de códigos que usa el resto del notebook — y con la que
se descubrió que **el mapeo anterior tenía masculino y femenino intercambiados**.

**Lo que se ve:** es la brecha más marcada del dataset. Drama (+1,6), romance (+1,0) y boys
love (+0,7) por encima de la media en el grupo femenino; acción (+0,9), ecchi (+0,7) y
ciencia ficción (+0,7) en el masculino. El grupo no binario se separa de los dos: award
winning, slice of life y boys love arriba, romance y ecchi abajo.

Las desviaciones son de uno o dos puntos porcentuales sobre repartos del 10-16%: son
diferencias reales y consistentes, no abismos.


In [ ]:
def chart_categoria_por_genero_usuario():
    """Preferencia de género según el género declarado del usuario, censo de 2023.

    En este volcado el género del usuario viene **como texto** (`Male`, `Female`,
    `Non-Binary`), así que aquí no hay ningún código que interpretar. Es además la
    fuente con la que se dedujo el mapeo de códigos que usa el resto del notebook.
    """
    desv = _grafico_desviacion(
        "Gender",
        orden=["Male", "Female", "Non-Binary"],
        etiquetas={"Male": "Masculino", "Female": "Femenino", "Non-Binary": "No binario"},
        titulo="Drama y romance frente a acción y ecchi: la brecha más marcada del dataset\n"
               "Desviación de cada género declarado frente al reparto general",
        pie="Censo completo del volcado de 2023: 224.383 perfiles con género declarado y 48,6 M de "
            "valoraciones.\nAzul, el grupo ve ese género más que la media; rojo, menos. El género "
            "del usuario viene como texto en esta fuente, así que no hay ningún código que "
            "interpretar.",
        nombre_fichero="categoria_por_genero.png")
    print(desv.round(2).to_string())

chart_categoria_por_genero_usuario()


## 8.5 Evolución de la nota media por año de estreno

**La versión anterior advertía de un confusor; esta lo comprueba.**

El texto decía que la subida de la nota con el año de estreno podía ser supervivencia y no
mejora: los títulos antiguos que siguen recibiendo votos son un público que los buscó a
propósito. Esa advertencia sigue siendo válida, pero con el panel de nueve cortes se puede
verificar algo distinto y comprobable: **si el patrón depende del momento en que se mide**.

La respuesta es que no. La curva se dibuja cuatro veces —medida en 2018, 2021, 2024 y 2026—
y las tres que usan la nota publicada por MyAnimeList **se solapan casi punto por punto**. La
distancia entre los años noventa y los veinte se mantiene en torno a 0,45 puntos en todos los
cortes. El de 2026 queda por debajo de forma sistemática, pero eso es el desfase de método
ya conocido: usa una media calculada por nosotros, no la nota ponderada de MAL.

**Fuera el doble eje.** La versión anterior superponía la nota y el número de estrenos en dos
escalas distintas sobre el mismo plano, lo que insinúa una correlación que el gráfico no
mide. Ahora son dos paneles apilados que comparten el eje de tiempo.

El año de estreno se toma por consenso entre cortes: el de 2026 solo lo trae para el 65% de
los títulos y el de 2024 para el 97%.


In [ ]:
def chart_evolucion_tiempo(min_titulos=20, cortes=(2018, 2021, 2024, 2026)):
    """Nota media por año de estreno, medida desde varios cortes del panel.

    Dos cambios sobre la versión anterior:

    1. **Se puede comprobar el confusor, no solo advertirlo.** Antes había una sola
       línea y una nota diciendo que la subida podía ser supervivencia y no mejora.
       Con cuatro cortes separados por ocho años se ve si el patrón se sostiene:
       si la forma se repite, no es un efecto del momento en que se mide.
    2. **Fuera el doble eje.** La versión anterior superponía la nota y el número de
       estrenos en dos escalas distintas, lo que insinúa una correlación que el
       gráfico no mide. Ahora son dos paneles con el mismo eje de tiempo.

    El año de estreno se toma por consenso entre cortes: el de 2026 solo cubre el
    65% de los títulos y el de 2024 el 97%.
    """
    panel = load_panel(list(cortes))
    anio = pd.DataFrame({a: d["year_estreno"] for a, d in panel.items()}).median(axis=1).round()
    anio = anio[anio.between(1960, SNAPSHOT_YEAR)]

    conteo = anio.value_counts().sort_index()
    validos = conteo[conteo >= min_titulos].index

    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True,
                                  gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08})
    for color, a in zip(RAMPA_CORTES, cortes):
        s = pd.DataFrame({"y": anio, "s": panel[a]["score"]}).dropna()
        serie = s[s["y"].isin(validos)].groupby("y")["s"].mean()
        ax.plot(serie.index, serie.values, color=color, linewidth=2,
                solid_capstyle="round", label=f"medido en {a}")
        ax.annotate(f" {a}", (serie.index[-1], serie.iloc[-1]), color=INK2 if False else "#52514e",
                    fontsize=9, va="center", fontweight="bold")
    ax.set_ylabel("Nota media de los títulos estrenados ese año (1-10)", fontsize=10)
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
    for e in ("top", "right"):
        ax.spines[e].set_visible(False)
    ax.legend(frameon=False, loc="upper left", fontsize=9, ncol=4)
    ax.set_title("La forma de la curva se repite en los cuatro cortes: no es un artefacto\n"
                 "Nota media por año de estreno, medida en 2018, 2021, 2024 y 2026",
                 fontsize=14, fontweight="bold", loc="left", pad=16)

    c = conteo[conteo.index.isin(validos)]
    ax2.bar(c.index, c.values, color="#c3c2b7", width=0.85)
    ax2.set_ylabel("Títulos\nestrenados", fontsize=9)
    ax2.set_xlabel("Año de estreno", fontsize=10)
    ax2.yaxis.grid(True)
    ax2.set_axisbelow(True)
    for e in ("top", "right"):
        ax2.spines[e].set_visible(False)

    ax2.text(0, -0.42, "Solo años con al menos {} títulos. El año de estreno se toma por consenso "
             "entre cortes.\nLos tres cortes con nota publicada por MyAnimeList (2018, 2021 y "
             "2024) se solapan casi punto por punto.\nEl de 2026 queda por debajo porque usa una "
             "media calculada por nosotros, no la nota ponderada de MAL: es un desfase de "
             "método,\nno una caída de la calidad.".format(min_titulos),
             transform=ax2.transAxes, fontsize=8, color="#898781", va="top")
    savefig("evolucion_tiempo_top10.png", fig)

    resumen = pd.DataFrame({a: pd.DataFrame({"y": anio, "s": panel[a]["score"]}).dropna()
                            .assign(dec=lambda x: (x["y"] // 10 * 10))
                            .groupby("dec")["s"].mean() for a in cortes})
    print("Nota media por década de estreno, según el corte desde el que se mide:")
    print(resumen.loc[resumen.index >= 1980].round(3).to_string())

chart_evolucion_tiempo()


## 8.6 Joyas ocultas

**Este gráfico también estaba invertido.** La versión anterior ordenaba por
`nota × log(audiencia)`, una fórmula que **premia la audiencia**: devolvía los éxitos
masivos, exactamente lo contrario de una joya oculta.

**La definición corregida** exige tres condiciones a la vez:

1. **Nota alta**, con un mínimo de `MIN_SCORED` votos para que sea comparable.
2. **Audiencia baja**: por debajo del percentil 60 de interacciones **del propio
   catálogo**, no de un número absoluto elegido a ojo. Así el umbral se adapta a los datos
   y sale impreso en el gráfico.
3. **Que sea una primera temporada.** Una cuarta temporada con nota alta y poca audiencia
   no es un descubrimiento: es el público residual de una serie que ya conoces, y su nota
   está inflada porque solo la puntúa quien llegó hasta ahí. El filtro sale de
   `anime_relaciones.parquet` —se descarta lo que tenga una relación `PREQUEL` (secuela) o
   `PARENT` (especial, recopilatorio, spin-off)— y no de buscar «Season 2» en el título,
   que falla con `Gintama'`, `Shippuuden` o los numerales romanos.

El orden es por nota, sin mezclar audiencia en la fórmula. La audiencia ya ha cumplido su
función como filtro; volver a meterla en el criterio de orden es lo que rompía el gráfico
anterior.

El resultado son títulos con notas de 8,5 a 8,8 y entre 4.000 y 24.000 interacciones,
frente a los 4 millones de los grandes éxitos.


In [ ]:
def chart_joyas_ocultas(top_n=12, min_scored=MIN_SCORED, percentil_audiencia=0.60,
                        solo_primera_temporada=True):
    """Joyas ocultas: nota alta con audiencia baja, entre primeras temporadas.

    Dos criterios que la versión original no tenía:

    1. Ordenaba por nota * log(audiencia), que premia justamente a los títulos más
       vistos: devolvía los éxitos masivos, lo contrario de una joya oculta.
    2. No distinguía continuaciones. Una cuarta temporada con nota alta y poca
       audiencia no es un descubrimiento: es el resto de público de una serie que
       ya conoces. Con `solo_primera_temporada` se excluyen las secuelas y los
       derivados usando `anime_relaciones.parquet`.
    """
    anime = load_anime_catalogo()[["mal_id", "title", "year"]]
    stats = load_stats_globales()[["anime_id", "n_interactions", "n_scored", "mean_score"]]
    m = stats.merge(anime, left_on="anime_id", right_on="mal_id", how="inner")

    if solo_primera_temporada:
        primeras = ids_primera_temporada()
        antes = len(m)
        m = m[m["mal_id"].isin(primeras)]
        print(f"solo primeras temporadas: {len(m):,} de {antes:,} títulos "
              f"({antes - len(m):,} secuelas o derivados descartados)")

    # Minimo de votos para que la nota sea comparable, y techo de audiencia
    # definido por un percentil del propio catalogo en vez de un numero arbitrario.
    elegibles = m[m["n_scored"] >= min_scored].copy()
    techo = elegibles["n_interactions"].quantile(percentil_audiencia)
    joyas = (
        elegibles[elegibles["n_interactions"] <= techo]
        .nlargest(top_n, "mean_score")
        .sort_values("mean_score")
    )
    print(f"umbral de audiencia (percentil {percentil_audiencia:.0%}): {techo:,.0f} interacciones")

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(joyas["title"], joyas["mean_score"], color="#E45756")
    ax.set_title(
        f"Top {top_n} joyas ocultas: nota alta con audiencia baja\n"
        f"(mín. {min_scored:,} votos, ≤ {techo:,.0f} interacciones"
        + (", solo primeras temporadas)" if solo_primera_temporada else ")")
    )
    ax.set_xlabel("Nota media de los usuarios (1-10)")
    ax.set_ylabel("Anime")
    ax.set_xlim(0, 12.6)
    ax.set_xticks(range(0, 11, 2))
    for y, (s, n) in enumerate(zip(joyas["mean_score"], joyas["n_interactions"])):
        ax.text(s + 0.1, y, f"{s:.2f} · {fmt_miles(n)} inter.", va="center", fontsize=9)
    savefig("joyas_ocultas.png", fig)
    print(joyas[["title", "mean_score", "n_scored", "n_interactions"]].iloc[::-1].to_string(index=False))

chart_joyas_ocultas()


# PREPARACIÓN DE LAS TABLAS ANALÍTICAS

Deriva de `clean/` un conjunto reducido de tablas en `clean/analysis/`, pensadas para
consultarse rápido y repetidamente durante el análisis.

**Qué se produce**

| Tabla | Filas | Contenido |
|---|---|---|
| `df_users.parquet` | 18,2 M | Subconjunto de columnas del perfil |
| `df_anime.parquet` | 19.384 | Catálogo con `title` resuelto |
| `df_anime_resumen.parquet` | 29.604 | Agregados por anime unidos al catálogo |
| `df_status_counts.parquet` | 9 | Reparto de estados sobre las interacciones |

**Qué no se produce, y por qué.** No hay tabla usuario-anime materializada. Con 527
millones de filas, cualquier intento de crearla agota la memoria. La versión anterior de
esta celda lo intentaba con `pd.read_parquet` sobre el fichero entero y no podía
completarse en ninguna máquina razonable.


## 9.1 Comprobación de requisitos

Verifica que existen las cuatro tablas de las que depende todo el bloque y **falla
inmediatamente** con un mensaje claro si falta alguna.

El motivo es práctico: sin esta comprobación, la ausencia de una tabla se manifestaría
tres celdas más abajo como un error de fichero no encontrado en mitad de un cálculo largo.
Es preferible detenerse aquí indicando qué falta y qué etapa hay que ejecutar.


In [ ]:
# Preparación del estudio: tablas analíticas derivadas de los parquet limpios.
#
# interacciones.parquet tiene ~527M de filas y 8.4 GB. Cargarlo con pd.read_parquet
# agota la memoria de la máquina, así que todo lo que dependa de él se agrega
# recorriendo el fichero por row group y guardando solo el resultado agregado.

REQUERIDOS = {
    "anime": CLEAN / "anime.parquet",
    "users": CLEAN / "users.parquet",
    "interacciones": CLEAN / "interacciones.parquet",
    "anime_stats": CLEAN / "anime_stats_globales.parquet",
}
for nombre, ruta in REQUERIDOS.items():
    print(f"{nombre:14} {'OK  ' if ruta.exists() else 'FALTA'} -> {ruta.name}")
faltan = [n for n, r in REQUERIDOS.items() if not r.exists()]
if faltan:
    raise FileNotFoundError(f"Faltan tablas limpias: {faltan}. Ejecuta antes las etapas 1-3.")


## 9.2 Tabla de perfiles para análisis

Copia de `users.parquet` con las columnas que el análisis necesita: de 49 columnas a 21.

**Se copia row group a row group**, sin materializar los 18,2 millones de filas en pandas.
El `ParquetWriter` se crea con el esquema del primer bloque leído y escribe los siguientes
detrás, lo que mantiene el consumo de memoria en un bloque cada vez.

**Las columnas que se conservan y por qué**

- Identificación y demografía: `username`, `gender`, `location`, `birth_*`.
- Actividad declarada: `anime_days`, `anime_completed`, `anime_inprogress`,
  `anime_dropped`, `anime_planned`, `anime_episodes`.
- Nota declarada: `anime_mean` **con su bandera `anime_mean_invalid`**.
- Contexto: `joined_date`, `last_online_ts`, `account_age_days`, `fav_anime_n`,
  `list_fetched_ok`.

**Las banderas de validación viajan con el dato.** Sin `anime_mean_invalid`, la tabla
analítica perdería la información de qué medias son imposibles y el filtrado tendría que
volver a deducirse. Esta era una de las causas de que las celdas de análisis fallaran: la
versión anterior seleccionaba un conjunto de columnas que no incluía `anime_completed`, y
las celdas siguientes lo pedían.

La lista de columnas se filtra por disponibilidad y avisa de las que falten, de modo que
la celda sigue funcionando si `users.parquet` procede de una versión anterior del pipeline.


In [ ]:
COLS_USERS = [
    "username", "gender", "location", "joined_date", "last_online_ts",
    "birth_year", "birth_month", "birth_day",
    "anime_mean", "anime_mean_invalid", "anime_days", "anime_days_invalid",
    "anime_inprogress", "anime_completed", "anime_onhold", "anime_dropped",
    "anime_planned", "anime_episodes",
    "fav_anime_n", "list_fetched_ok", "account_age_days",
]


def preparar_df_users():
    """Copia por row group el subconjunto de columnas de perfil que usa el análisis."""
    src = pq.ParquetFile(CLEAN / "users.parquet")
    disponibles = [c for c in COLS_USERS if c in src.schema_arrow.names]
    faltantes = [c for c in COLS_USERS if c not in disponibles]
    if faltantes:
        print(f"  aviso: columnas ausentes en users.parquet -> {faltantes}")
    destino = ANALYSIS / "df_users.parquet"
    writer, filas = None, 0
    try:
        for rg in range(src.metadata.num_row_groups):
            tb = src.read_row_group(rg, columns=disponibles)
            if writer is None:
                writer = pq.ParquetWriter(destino, tb.schema, compression="zstd")
            writer.write_table(tb)
            filas += tb.num_rows
    finally:
        if writer is not None:
            writer.close()
    print(f"  df_users.parquet: {filas:,} filas x {len(disponibles)} columnas")
    return destino


## 9.3 Catálogo y resumen por anime

**`df_anime.parquet`** — el catálogo con el `title` ya resuelto y los nombres de columna
normalizados.

**`df_anime_resumen.parquet`** — une los agregados de la etapa 3 con el catálogo mediante
un `left join` desde los agregados. La dirección importa: la tabla base son los títulos
**observados en las interacciones** (29.604), no los del catálogo (19.384). Un `inner
join` descartaría en silencio los 10.302 títulos vistos por usuarios pero ausentes de
AniList.

Esos títulos se marcan con `sin_metadata` y su nombre se rellena con `mal_id N`. Así
aparecen en los recuentos con una etiqueta honesta en lugar de desaparecer o mostrar una
celda vacía.

**Esta tabla es la que rompía el análisis.** La versión anterior se construía sin las
columnas `title` ni `popularity`, y tres celdas más abajo se pedían: el bloque fallaba con
`KeyError`. Aquí se incluyen ambas explícitamente.

El resultado se ordena por número de interacciones, que es el orden en el que se consulta
casi siempre.


In [ ]:
def preparar_df_anime():
    a = load_anime_catalogo()
    cols = ["mal_id", "title", "title_english", "title_romaji", "title_native",
            "type", "format", "episodes", "average_score", "meanScore",
            "popularity", "favourites", "year"]
    df = a[[c for c in cols if c in a.columns]].copy()
    df.to_parquet(ANALYSIS / "df_anime.parquet", index=False)
    print(f"  df_anime.parquet: {len(df):,} filas x {df.shape[1]} columnas")
    return df


def preparar_df_anime_resumen(df_anime):
    stats = load_stats_globales()
    resumen = stats.rename(columns={"anime_id": "mal_id"}).merge(df_anime, on="mal_id", how="left")
    resumen["title"] = resumen["title"].fillna("mal_id " + resumen["mal_id"].astype(str))
    resumen["sin_metadata"] = resumen["type"].isna() if "type" in resumen.columns else True
    resumen = resumen.sort_values("n_interactions", ascending=False).reset_index(drop=True)
    resumen.to_parquet(ANALYSIS / "df_anime_resumen.parquet", index=False)
    print(f"  df_anime_resumen.parquet: {len(resumen):,} filas x {resumen.shape[1]} columnas "
          f"({int(resumen['sin_metadata'].sum()):,} sin metadata de AniList)")
    return resumen


## 9.4 Reparto de estados sobre las interacciones

El único cálculo de esta sección que **recorre el fichero grande**. Lee solo la columna
`status` —un `int8`— de los 924 row groups y acumula el recuento bloque a bloque.

**Coste:** unos 8 minutos sobre los 527 millones de filas. `max_row_groups` permite
limitarlo durante las pruebas.

**El resultado deja constancia de su propio alcance**: las columnas `row_groups_leidos` y
`row_groups_totales` acompañan a los datos, de modo que una tabla calculada sobre una
muestra no puede confundirse después con una calculada sobre el total.

Los códigos sin etiqueta conocida se agrupan como `desconocido` **conservando su código
numérico** en la columna `status`, para poder investigarlos. Aparecen unas 33.800
filas con código 0 y un puñado con códigos 5, 33 y 55.


In [ ]:
STATUS_LABELS_VIZ = {1: "watching", 2: "completed", 3: "on_hold", 4: "dropped", 6: "plan_to_watch"}


def preparar_df_status_counts(max_row_groups=None):
    """Distribución de estados sobre interacciones.parquet, agregando por bloques."""
    pf = pq.ParquetFile(CLEAN / "interacciones.parquet")
    total_rgs = pf.metadata.num_row_groups
    n = total_rgs if max_row_groups is None else min(max_row_groups, total_rgs)
    print(f"  recorriendo {n}/{total_rgs} row groups de interacciones "
          f"({pf.metadata.num_rows:,} filas en total)...")
    acumulado = None
    for rg in range(n):
        s = pf.read_row_group(rg, columns=["status"]).column("status").to_pandas()
        c = s.value_counts(dropna=False)
        acumulado = c if acumulado is None else acumulado.add(c, fill_value=0)
    df = acumulado.rename_axis("status").reset_index(name="count")
    df["status_label"] = df["status"].map(STATUS_LABELS_VIZ).fillna("desconocido")
    df["count"] = df["count"].astype("int64")
    df["pct"] = (100 * df["count"] / df["count"].sum()).round(4)
    df = df.sort_values("count", ascending=False).reset_index(drop=True)
    df["row_groups_leidos"] = n
    df["row_groups_totales"] = total_rgs
    df.to_parquet(ANALYSIS / "df_status_counts.parquet", index=False)
    print(f"  df_status_counts.parquet: {len(df)} filas")
    return df


## 9.5 Ejecución y primeros resultados

Lanza las cuatro preparaciones e imprime dos resúmenes de control.

**Reparto de estados esperado**: *completed* domina con ~70%, seguido de *plan to watch*
con ~17%. Es coherente con el comportamiento de una lista de seguimiento y sirve como
comprobación de que el filtro de cohorte no ha distorsionado la distribución.

**Top 10 por interacciones**: Attack on Titan, Death Note, Fullmetal Alchemist:
Brotherhood… Es el resultado que uno esperaría, y por eso funciona como validación de
extremo a extremo: si el cruce entre `anime_id` y el catálogo estuviera desplazado, aquí
saldrían títulos arbitrarios.

Ejecutar esta celda entera lleva unos 8 minutos, casi todos en el recuento de estados.


In [ ]:
print("\nPreparando tablas analíticas...")
preparar_df_users()
df_anime = preparar_df_anime()
df_anime_resumen = preparar_df_anime_resumen(df_anime)
# max_row_groups=None recorre el fichero completo (varios minutos).
df_status_counts = preparar_df_status_counts(max_row_groups=None)

print("\nDistribución de estados en la cohorte:")
print(df_status_counts[["status_label", "count", "pct"]].to_string(index=False))

print("\nTop 10 anime por interacciones:")
print(df_anime_resumen[["title", "n_interactions", "n_scored", "mean_score"]].head(10).to_string(index=False))

print(f"\nTablas guardadas en: {ANALYSIS}")
print("Nota: no se materializa una tabla usuario-anime completa; con 527M de filas el "
      "análisis a nivel de interacción debe hacerse con iter_interacciones().")


# ANÁLISIS

Cuatro bloques sobre las tablas de `clean/analysis/`: perfil general, nota por género,
relación entre popularidad y nota, y segmentación de usuarios por volumen de consumo.

**Todas las conclusiones que se imprimen se calculan de los datos.** La versión anterior
del notebook tenía frases con cifras escritas a mano dentro de los `print` («abandonan ~42
animes», «nota entre 6.8 y 7.3»), que dejan de ser ciertas en cuanto cambia el dataset y
no hay forma de notarlo.


## 10.1 Perfil general de los usuarios

Primera lectura de `df_users`, cargando solo ocho columnas.

**El filtro de nota es el punto crítico.** `nota_media_valida()` descarta las medias en
cero —perfiles que no han puntuado nada— y las marcadas como fuera de escala. La celda
imprime cuántos ceros se han excluido para que la magnitud del filtro sea visible: son
10,4 millones de perfiles.

**Definición de usuario activo:** al menos un anime completado. Son 8,3 millones de 18,2
(45,7%). El resto son cuentas creadas y abandonadas, algo habitual en cualquier plataforma
con registro abierto.

El top 10 por animes completados sirve como control de valores extremos: aparecen perfiles
con más de 28.000 títulos completados, que son con casi total seguridad cuentas
automatizadas o importaciones masivas. No se eliminan —el criterio del pipeline es no
tocar el dato— pero conviene saber que están ahí antes de calcular cualquier máximo.


In [ ]:
# Exploración inicial: perfiles, géneros y calidad del dato
df_users = pd.read_parquet(
    ANALYSIS / "df_users.parquet",
    columns=["username", "anime_mean", "anime_mean_invalid", "anime_days",
             "anime_completed", "anime_dropped", "anime_planned", "list_fetched_ok"],
)
df_anime_resumen = pd.read_parquet(ANALYSIS / "df_anime_resumen.parquet")
anime_generos = pd.read_parquet(CLEAN / "anime_generos.parquet")
print(f"df_users: {df_users.shape}   df_anime_resumen: {df_anime_resumen.shape}   "
      f"anime_generos: {anime_generos.shape}")

# --- Resumen de usuarios ---
# Se excluyen las medias fuera de escala (etapa 2) y las medias en cero, que en el
# origen significan "el usuario no ha puntuado nada", no una valoracion de 0.
mean_valida = nota_media_valida(df_users)
ceros = int((df_users["anime_mean"] == 0).sum())
usuarios_con_dias = df_users["anime_days"].notna().sum()
print(f"\nUsuarios totales: {len(df_users):,}")
print(f"Usuarios con anime_days informado: {usuarios_con_dias:,} "
      f"({100*usuarios_con_dias/len(df_users):.1f}%)")
print(f"Usuarios con al menos un anime completado: {int((df_users['anime_completed'] > 0).sum()):,}")
print("\nDistribución de anime_days:")
print(df_users["anime_days"].describe().to_string())
print(f"\nUsuarios con anime_mean = 0 (sin ninguna puntuación): {ceros:,}")
print("\nDistribución de la nota media por usuario (excluyendo ceros y medias fuera de escala):")
print(mean_valida.describe().to_string())

print("\nTop 10 usuarios por animes completados:")
print(
    df_users.nlargest(10, "anime_completed")[["username", "anime_completed", "anime_days", "anime_mean"]]
    .to_string(index=False)
)


## 10.2 Resumen del catálogo y de los géneros

Dos listas de control sobre `df_anime_resumen` y `anime_generos`.

**Top 10 por interacciones** con el reparto por estado al lado. La comparación entre
`n_completed` y `n_dropped` es más informativa que el total: dos títulos con volumen
parecido pueden tener tasas de abandono muy distintas, y ahí está la señal de calidad
percibida.

**Top 15 de géneros por número de títulos.** Sirve para decidir qué géneros tienen masa
suficiente para analizarse por separado: los que caen por debajo de unas decenas de
títulos no sostienen una media estable, y por eso el análisis por género de la celda
siguiente exige un mínimo de 20.


In [ ]:
# --- Resumen de animes ---
cols_resumen = [c for c in ["title", "n_interactions", "mean_score", "n_completed",
                            "n_plan_to_watch", "n_dropped"] if c in df_anime_resumen.columns]
print("\nTop 10 animes por interacciones:")
print(df_anime_resumen[cols_resumen].head(10).to_string(index=False))

# --- Top generos ---
top_generos = anime_generos["genre"].value_counts().head(15)
print("\nTop 15 géneros por número de títulos:")
print(top_generos.to_string())


## 10.3 Gráficos de resumen

Dos vistas de control, con la escala del eje formateada por `fmt_miles` según la magnitud
real de los datos.

- **Top 10 de géneros por número de títulos**: cuánto catálogo hay de cada tipo.
- **Top 10 de animes por interacciones**: cuánto se consume realmente.

Las dos juntas muestran que **cantidad de catálogo y volumen de consumo no coinciden**:
el género con más títulos no es el que acumula más actividad.


In [ ]:
# --- Graficos ---
top10_generos = top_generos.head(10).sort_values()
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top10_generos.index, top10_generos.values, color="#4C78A8")
ax.set_title("Top 10 géneros por número de títulos")
ax.set_xlabel("Número de animes")
ax.set_ylabel("Género")
savefig("top_generos.png", fig)

top_anime = df_anime_resumen.nlargest(10, "n_interactions").iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top_anime["title"], top_anime["n_interactions"], color="#F58518")
ax.set_title("Top 10 animes por número de interacciones")
ax.set_xlabel("Número de interacciones")
ax.set_ylabel("Anime")
ax.xaxis.set_major_formatter(plt.FuncFormatter(fmt_miles))
savefig("top_animes_interacciones.png", fig)

print("\nSiguientes pasos: 1) nota por género  2) popularidad frente a nota  "
      "3) segmentación de usuarios por consumo")


## 10.4 Nota media por género, ponderada por votos

**Por qué ponderar.** La media simple de las notas de un género trata igual a un título
con 2 millones de votos y a uno con mil. La media ponderada por `n_scored` responde a una
pregunta distinta y más útil: *¿qué nota recibe, de media, una hora de visionado de este
género?*

La diferencia entre ambas cifras es grande y las dos se imprimen y se anotan en el
gráfico: la ponderada ronda el 7,7-8,0 y la simple el 6,4-7,1. **La ponderada es más alta
en todos los géneros**, lo que indica que dentro de cada género los títulos más vistos
están mejor valorados que la media de títulos del género.

**Cómo se calcula.** De forma vectorizada: `sum(nota × votos) / sum(votos)` mediante una
columna auxiliar y un `groupby().agg()`. La versión anterior usaba `groupby().apply()` con
una función Python por grupo, más lenta y afectada por el cambio de comportamiento de
`apply` en pandas 2.2+ respecto a las columnas de agrupación.

**Cuidado al comparar con el análisis de géneros de otros años.** Este gráfico usa la
taxonomía de **AniList**; los volcados de 2022 en adelante usan la de **MyAnimeList**, y solo
14 etiquetas coinciden entre las dos. AniList tiene *Thriller, Psychological, Mecha, Music,
Mahou Shoujo*; MyAnimeList tiene *Suspense, Award Winning, Gourmet, Boys Love, Girls Love,
Erotica, Avant Garde*. Por eso aquí el género mejor valorado sale *Thriller* y en un análisis
sobre datos de MyAnimeList saldría *Suspense*: **son casi el mismo concepto con dos nombres**,
no una discrepancia entre fuentes.

**Mínimo de 20 títulos por género**, por el mismo motivo que los umbrales de audiencia: un
género con cuatro títulos no sostiene una media comparable.


In [ ]:
# Nota por género y relación entre popularidad y nota media
anime = pd.read_parquet(ANALYSIS / "df_anime_resumen.parquet")
anime_generos = pd.read_parquet(CLEAN / "anime_generos.parquet")

for c in ["mean_score", "n_scored", "popularity", "n_interactions"]:
    anime[c] = pd.to_numeric(anime[c], errors="coerce")

# --- 1) Nota media por género, ponderada por número de votos ---
merged = anime_generos.merge(
    anime[["mal_id", "title", "mean_score", "n_scored", "popularity", "n_interactions"]],
    on="mal_id", how="inner",
).dropna(subset=["genre", "mean_score"])
merged["n_scored"] = merged["n_scored"].fillna(0)

# La media ponderada se calcula vectorizada: sum(nota * votos) / sum(votos).
merged["_num"] = merged["mean_score"] * merged["n_scored"]
agg = merged.groupby("genre").agg(
    animes=("mal_id", "nunique"),
    avg_score=("mean_score", "mean"),
    _num=("_num", "sum"),
    _den=("n_scored", "sum"),
    avg_popularity=("popularity", "mean"),
    avg_interactions=("n_interactions", "mean"),
).reset_index()
agg["weighted_score"] = np.where(agg["_den"] > 0, agg["_num"] / agg["_den"].where(agg["_den"] > 0), np.nan)
genre_summary = (
    agg[agg["animes"] >= 20]
    .drop(columns=["_num", "_den"])
    .sort_values("weighted_score", ascending=False)
    .head(12)
)

print("Top géneros por nota ponderada (mínimo 20 títulos):")
print(genre_summary[["genre", "animes", "avg_score", "weighted_score"]].to_string(index=False))

plot_gs = genre_summary.iloc[::-1]
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(plot_gs["genre"], plot_gs["weighted_score"], color="#2E86DE")
for i, (w, a) in enumerate(zip(plot_gs["weighted_score"], plot_gs["avg_score"])):
    ax.text(w + 0.05, i, f"pond. {w:.2f} · simple {a:.2f}", va="center", fontsize=9)
ax.set_title("Top 12 géneros por nota media ponderada por votos", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Nota media ponderada (1-10)", fontsize=11)
ax.set_ylabel("Género", fontsize=11)
ax.set_xlim(0, genre_summary["weighted_score"].max() * 1.35)
savefig("score_por_genero.png", fig)


## 10.5 Popularidad frente a nota media

**La audiencia y la nota ahora salen de la misma pasada.** La versión anterior cruzaba
`popularity` de AniList (captura de agosto de 2026) con la nota derivada de las listas de
MyAnimeList (febrero de 2026): dos plataformas y seis meses de diferencia para responder una
sola pregunta. Ahora la audiencia es `n_interactions`, calculada sobre la misma población y
en la misma pasada que la nota.

**La conclusión no cambia**, que es justo lo que se quería comprobar: medido sobre los mismos
9.534 títulos, la correlación de rangos es 0,539 con la popularidad de AniList, 0,510 con los
miembros de MyAnimeList de 2024 y 0,525 con la audiencia propia. Lo que cambia es que el
gráfico deja de mezclar orígenes.

El eje se dibuja en **escala logarítmica** porque la distribución abarca varios órdenes de
magnitud; en escala lineal el 99% de los puntos se apelotonaría contra el eje.

**La correlación se calcula sobre rangos** (Spearman), no sobre los valores: la relación
entre popularidad y nota no es lineal, y Spearman mide si el orden se mantiene, que es la
pregunta pertinente. Se calcula como la correlación de Pearson sobre los rangos, sin
depender de scipy.

**Resultado: ρ ≈ 0,54**, una relación positiva moderada. Lo popular tiende a estar mejor
valorado, pero la dispersión es amplia: hay títulos muy vistos con notas mediocres y
títulos poco vistos con notas altas —los de la celda 8.6—.

Solo entran los títulos con al menos `MIN_SCORED` votos, y la muestra de 2.000 puntos del
gráfico es para legibilidad; **la correlación se calcula sobre el conjunto completo**, no
sobre la muestra. Los diez títulos destacados son los de mayor volumen entre los que
superan media + 0,5 desviaciones.


In [ ]:
# --- 2) Audiencia frente a nota media ---
# La versión anterior cruzaba `popularity` de AniList (captura de agosto de 2026)
# con la nota derivada de las listas de MyAnimeList (febrero de 2026): dos
# plataformas y seis meses de diferencia para responder una sola pregunta.
# Ahora la audiencia sale de `n_interactions`, que se calcula en la misma pasada y
# sobre la misma población que la nota. La conclusión no cambia —medido sobre 9.534
# títulos, la correlación de rangos es 0,539 con popularity de AniList, 0,510 con
# los miembros de MyAnimeList de 2024 y la de abajo con la audiencia propia— pero
# el gráfico deja de mezclar orígenes.
scatter = anime.dropna(subset=["mean_score", "n_interactions", "n_scored"])
scatter = scatter[scatter["n_scored"] >= MIN_SCORED]
muestra = scatter.sample(min(len(scatter), 2000), random_state=42)
# Spearman es la correlación de Pearson sobre los rangos: se calcula así para no
# depender de scipy, que pandas exige al pasar method="spearman".
corr = scatter["n_interactions"].rank().corr(scatter["mean_score"].rank())
print(f"\nTítulos con al menos {MIN_SCORED:,} votos: {len(scatter):,}")
print(f"Correlación de Spearman entre audiencia y nota media: {corr:.3f}")

media_nota = muestra["mean_score"].mean()
std_nota = muestra["mean_score"].std()

fig, ax = plt.subplots(figsize=(13, 8))
ax.scatter(muestra["n_interactions"], muestra["mean_score"], s=20, alpha=0.45,
           c=muestra["mean_score"], cmap="viridis")
destacados = muestra[muestra["mean_score"] > media_nota + 0.5 * std_nota].nlargest(10, "n_interactions")
for _, row in destacados.iterrows():
    ax.annotate(
        row["title"][:25],
        xy=(row["n_interactions"], row["mean_score"]),
        xytext=(10, 10), textcoords="offset points", fontsize=8, alpha=0.85,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFF3B0", alpha=0.6),
        arrowprops=dict(arrowstyle="->", alpha=0.5),
    )
ax.axhline(media_nota, color="red", linestyle="--", linewidth=2, alpha=0.6,
           label=f"Nota media de la muestra ({media_nota:.2f})")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(plt.FuncFormatter(fmt_miles))
ax.set_title(
    f"Audiencia frente a nota media (ρ de Spearman = {corr:.2f})\n"
    f"Muestra de {len(muestra):,} títulos con al menos {MIN_SCORED:,} votos",
    fontsize=14, fontweight="bold", pad=15,
)
ax.set_xlabel("Audiencia: interacciones registradas en la cohorte (escala log)", fontsize=11)
ax.set_ylabel("Nota media de los usuarios (1-10)", fontsize=11)
ax.legend(loc="lower right", fontsize=10)
savefig("popularidad_vs_nota.png", fig)

print("\nLas joyas ocultas se calculan en la celda de gráficos anterior "
      "(chart_joyas_ocultas), para no duplicar el mismo fichero de salida.")


## 10.6 Segmentación de usuarios por volumen de consumo

Divide a los usuarios activos en cinco tramos por animes completados: Casual (1-10),
Regular (11-50), Aficionado (51-100), Entusiasta (101-500) y Obsesivo (500+).

**Los cortes son convencionales**, elegidos para que los tramos tengan tamaños comparables
y significado intuitivo. No salen de un método de agrupamiento.

**Aquí se aplica la corrección de la nota en cero** y su efecto cambia una conclusión del
notebook. Con los ceros incluidos, la correlación entre animes completados y nota media
salía ≈ 0 y el notebook afirmaba —en un texto escrito a mano— que «los usuarios que ven
más anime no tienen criterios de calidad más altos o más bajos».

Excluyendo los ceros, la correlación es **-0,21** y la nota media **desciende de forma
monótona** a lo largo de los cinco tramos:

| Segmento | Nota media | Completados | Abandonados |
|---|---|---|---|
| Casual (1-10) | 8,70 | 3,9 | 0,2 |
| Regular (11-50) | 8,30 | 27,1 | 1,2 |
| Aficionado (51-100) | 8,05 | 72,4 | 3,5 |
| Entusiasta (101-500) | 7,82 | 208,2 | 11,5 |
| Obsesivo (500+) | 7,46 | 825,0 | 42,1 |

Quien más ve, más bajo puntúa. La lectura razonable es que el usuario ocasional solo
puntúa lo que ha elegido con cuidado, mientras que el intensivo cubre todo el rango de
calidad y usa la escala completa.

**Correlación no es causa**, y el descenso también puede venir del propio sesgo de
selección de lo que cada grupo llega a ver.


In [ ]:
# Análisis de perfiles de usuario: segmentación por volumen de consumo
df_users = pd.read_parquet(
    ANALYSIS / "df_users.parquet",
    columns=["username", "anime_mean", "anime_mean_invalid", "anime_days",
             "anime_completed", "anime_inprogress", "anime_dropped", "anime_planned"],
)
for col in ["anime_mean", "anime_days", "anime_completed", "anime_inprogress",
            "anime_dropped", "anime_planned"]:
    df_users[col] = pd.to_numeric(df_users[col], errors="coerce")

# Se descartan las medias fuera de escala y las medias en cero: en el origen, 0.00
# significa que el perfil no tiene ninguna puntuacion, no que puntue con un 0.
ceros = int((df_users["anime_mean"] == 0).sum())
df_users["anime_mean"] = nota_media_valida(df_users)
print(f"Perfiles con nota media en cero tratados como sin puntuar: {ceros:,}")

print("=== PERFIL DE USUARIOS ===")
print(f"Total de usuarios: {len(df_users):,}")

activos = df_users[df_users["anime_completed"] > 0].copy()
pct_activos = 100 * len(activos) / len(df_users)
print(f"Usuarios con al menos un anime completado: {len(activos):,} ({pct_activos:.1f}%)")

BINS = [0, 10, 50, 100, 500, np.inf]
ETIQUETAS = ["Casual (1-10)", "Regular (11-50)", "Aficionado (51-100)",
             "Entusiasta (101-500)", "Obsesivo (500+)"]
# Las etiquetas se guardan en una linea para que las tablas impresas se lean bien
# y se parten en dos solo al dibujarlas en los ejes.
def eje(labels):
    return [l.replace(" (", "\n(") for l in labels]
activos["segment"] = pd.cut(activos["anime_completed"], bins=BINS, labels=ETIQUETAS)

conteo = activos["segment"].value_counts().sort_index()
print("\nSegmentación por animes completados:")
print(conteo.to_string())

# La correlación se calcula sobre las filas donde ambas variables existen.
pareja = activos[["anime_completed", "anime_mean"]].dropna()
corr = pareja["anime_completed"].corr(pareja["anime_mean"])
print(f"\nCorrelación de Pearson (completados vs nota media) sobre {len(pareja):,} usuarios: {corr:.4f}")

resumen_seg = activos.groupby("segment", observed=True).agg(
    usuarios=("username", "size"),
    nota_media=("anime_mean", "mean"),
    nota_std=("anime_mean", "std"),
    completados=("anime_completed", "mean"),
    dias=("anime_days", "mean"),
    abandonados=("anime_dropped", "mean"),
)
print("\nResumen por segmento:")
print(resumen_seg.round(2).to_string())

COLORES = ["#3498DB", "#2ECC71", "#F39C12", "#E74C3C", "#9B59B6"]


## 10.7 Distribución y nota por segmento

**Distribución de usuarios.** Los márgenes de las etiquetas se calculan como porcentaje
del máximo de la serie, no con un desplazamiento fijo. El código anterior sumaba 30.000
unidades a la altura de la barra, un número que solo funciona con datos de esa magnitud
concreta.

**Nota media por segmento**, con la desviación típica como barra de error. La desviación
importa tanto como la media: el segmento Casual tiene la nota más alta (8,70) pero también
la mayor dispersión (1,31), porque muchos de sus perfiles puntúan muy pocos títulos y su
media es inestable. Los segmentos intermedios son mucho más consistentes (≈0,93).


In [ ]:
# === 1) Distribución de usuarios por segmento ===
fig, ax = plt.subplots(figsize=(11, 6))
barras = ax.bar(range(len(conteo)), conteo.values, color=COLORES[:len(conteo)])
ax.set_xticks(range(len(conteo)))
ax.set_xticklabels(eje(conteo.index), fontsize=10)
margen = conteo.max() * 0.02
for barra, valor in zip(barras, conteo.values):
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + margen,
            f"{valor:,.0f}\n({100*valor/len(activos):.1f}%)", ha="center", fontsize=9, fontweight="bold")
ax.set_ylim(0, conteo.max() * 1.18)
ax.set_title("Distribución de usuarios por segmento de consumo", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("Número de usuarios", fontsize=11)
ax.set_xlabel("Segmento", fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(fmt_miles))
savefig("usuarios_por_segmento.png", fig)

# === 2) Nota media por segmento ===
fig, ax = plt.subplots(figsize=(11, 6))
barras = ax.bar(range(len(resumen_seg)), resumen_seg["nota_media"].values,
                yerr=resumen_seg["nota_std"].values, capsize=5,
                color=COLORES[:len(resumen_seg)], alpha=0.85, error_kw={"linewidth": 2})
ax.set_xticks(range(len(resumen_seg)))
ax.set_xticklabels(eje(resumen_seg.index), fontsize=10)
ax.set_ylim(0, 10)
for barra, nota in zip(barras, resumen_seg["nota_media"].values):
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 0.2,
            f"{nota:.2f}", ha="center", fontsize=9, fontweight="bold")
ax.set_title("Nota media por segmento de usuario (barras de error: desviación típica)",
             fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("Nota media declarada (1-10)", fontsize=11)
ax.set_xlabel("Segmento", fontsize=11)
savefig("nota_por_segmento.png", fig)


## 10.8 Comportamiento por segmento y relación con la nota

**Comportamiento por segmento: tres paneles, no uno.** La versión anterior metía
completados, días vistos y abandonados en un solo eje multiplicando cada serie por una
constante distinta (`÷50`, `÷10`, `×10`). Eso hace que las alturas no signifiquen nada y
que la comparación visual sea engañosa. Con un panel por métrica cada una conserva su
escala real.

**Completados frente a nota**, con el eje x en escala logarítmica por la enorme asimetría
de la distribución. La recta de tendencia se ajusta sobre las **filas emparejadas** donde
existen ambas variables. La versión anterior pasaba a `np.polyfit` dos series con
`dropna()` aplicado por separado, de longitudes distintas — un error que impedía ejecutar
la celda.


In [ ]:
# === 3) Comportamiento por segmento ===
# Cada métrica tiene su propia escala, así que se usan tres paneles en vez de
# multiplicar los valores por constantes arbitrarias para meterlos en un solo eje.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (col, titulo, color) in zip(axes, [
    ("completados", "Animes completados", "#3498DB"),
    ("dias", "Días de anime vistos", "#2ECC71"),
    ("abandonados", "Animes abandonados", "#E74C3C"),
]):
    ax.bar(range(len(resumen_seg)), resumen_seg[col].values, color=color, alpha=0.9)
    ax.set_xticks(range(len(resumen_seg)))
    ax.set_xticklabels(resumen_seg.index, rotation=45, ha="right", fontsize=8)
    ax.set_title(titulo, fontsize=11, fontweight="bold")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(fmt_miles))
fig.suptitle("Comportamiento medio por segmento de usuario", fontsize=14, fontweight="bold")
savefig("comportamiento_por_segmento.png", fig)

# === 4) Completados frente a nota media ===
muestra = pareja.sample(min(5000, len(pareja)), random_state=42)
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(muestra["anime_completed"], muestra["anime_mean"], s=15, alpha=0.35,
           c=muestra["anime_completed"], cmap="plasma")
# polyfit exige vectores de la misma longitud: se ajusta sobre las filas emparejadas.
pendiente, ordenada = np.polyfit(muestra["anime_completed"], muestra["anime_mean"], 1)
x = np.linspace(muestra["anime_completed"].min(), muestra["anime_completed"].max(), 100)
ax.plot(x, pendiente * x + ordenada, "r--", linewidth=2, alpha=0.7,
        label=f"Tendencia (r={corr:.3f})")
ax.set_xscale("log")
ax.set_title("Animes completados frente a nota media declarada", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Animes completados (escala log)", fontsize=11)
ax.set_ylabel("Nota media declarada (1-10)", fontsize=11)
ax.legend(fontsize=10)
savefig("completados_vs_nota.png", fig)


## 10.9 Hallazgos

Cierre del análisis. **Todas las cifras se calculan de las tablas** en el momento de
imprimirse: si cambia el dataset, cambian los números.

Esto sustituye a un bloque de conclusiones con valores escritos a mano que ya no se
correspondían con los datos, entre ellos la afirmación sobre la ausencia de relación entre
volumen de consumo y nota, que la corrección de las medias en cero desmiente.

**Direcciones naturales de continuación**

1. **Temporal**: evolución del consumo usando `created_at` y `updated_at` de las
   interacciones, en streaming.
2. **Por estado**: qué distingue a los títulos con alta tasa de abandono, cruzando
   `n_dropped / n_interactions` con género y formato.
3. **Preferencias por segmento**: si los géneros preferidos cambian según el volumen de
   consumo del usuario.
4. **Red social**: `user_amigos.parquet`, recordando que el grafo está truncado a diez
   amistades por usuario y por tanto es parcial.


In [ ]:
# === Conclusiones calculadas, no escritas a mano ===
mayor = resumen_seg["usuarios"].idxmax()
print("\n=== HALLAZGOS ===")
print(f"- {len(activos):,} usuarios con actividad de {len(df_users):,} en total ({pct_activos:.1f}%).")
print(f"- Segmento más numeroso: {mayor} "
      f"({100*resumen_seg.loc[mayor, 'usuarios']/len(activos):.1f}% de los activos).")
print(f"- Correlación completados-nota: {corr:.4f} "
      f"({'sin relación apreciable' if abs(corr) < 0.1 else 'relación apreciable'}).")
print(f"- Nota media por segmento entre {resumen_seg['nota_media'].min():.2f} "
      f"y {resumen_seg['nota_media'].max():.2f}.")
print(f"- Animes abandonados de media: {resumen_seg['abandonados'].min():.1f} en el segmento más bajo "
      f"frente a {resumen_seg['abandonados'].max():.1f} en el más alto.")
